<a href="https://colab.research.google.com/github/sulucay01/multimodal-rs-segmentation/blob/dev/notebooks/05_phase3_ablations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 - Phase 3 Ablation Studies

DI725 Term Project. Multimodal Fusion for Remote Sensing Land Cover Segmentation.

This notebook extends the Phase 2 multimodal model (UNetFormer with a frozen CLIP text encoder and gated cross-attention) with two controlled ablation studies and a combined-model check. It builds on the vision-only baseline from `03_baseline_models.ipynb` and the multimodal baseline from `04_multimodal.ipynb`. Each ablation changes a single component relative to the Phase 2 multimodal model and holds everything else fixed.

**Research questions**
1. Does a text encoder pretrained on remote sensing data (RemoteCLIP) improve land cover segmentation over a general-purpose text encoder (CLIP), holding the architecture and caption fixed?
2. Does a region-based loss (Dice, Lovász, or Tversky, combined with weighted cross-entropy) improve segmentation over weighted cross-entropy alone, holding the encoder and caption fixed?
3. When the best encoder and the best loss are applied together, do their individual gains combine additively, or do they overlap?

**Ablation 1: Text encoder (RQ1).** The general-purpose CLIP text encoder is replaced with RemoteCLIP. Both share the `ViT-B-32` architecture, so only the encoder weights change. The swap is evaluated across all five caption variants.

**Ablation 2: Loss function (RQ2).** Weighted cross-entropy is replaced with a region-based composite loss (CE+Dice, CE+Lovász, or CE+Tversky), holding the encoder and caption fixed at the Phase 2 setup.

**Additivity check (RQ3).** A single model is trained with both ablation winners applied at once, the best text encoder and the best loss, to measure whether the two improvements combine additively.

Selection protocol: every configuration is checkpointed on the validation set, the best configuration in each study is selected by validation mIoU, and that configuration is then evaluated once on the held-out test set. The architecture matches `04_multimodal.ipynb` so that all comparisons stay consistent.

Sections:
1. Setup
2. Dataset and DataLoaders
3. UNetFormer architecture
4. Multimodal architecture
5. Training infrastructure
6. Ablation 1: text encoder
7. Ablation 2: loss function
8. Additivity check
9. Summary

## 1. Setup


In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install dependencies.
# open_clip_torch provides the frozen CLIP text encoder.
# transformers provides the RemoteCLIP weights.
!pip install transformers timm einops wandb open_clip_torch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.4 MB/s eta 0:00:00


In [3]:
# All imports for the notebook.
import numpy as np
import pandas as pd
import json
import time
from pathlib import Path
import shutil
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm
from timm.layers import DropPath, trunc_normal_
from einops import rearrange
import open_clip
from huggingface_hub import hf_hub_download

import logging
logging.getLogger('open_clip').setLevel(logging.ERROR)

import wandb

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


In [4]:
# Project paths
PROJECT_DIR     = Path('/content/drive/MyDrive/DI725_Project')
DATA_DIR_DRIVE  = PROJECT_DIR / 'data'
SPLITS_CSV      = DATA_DIR_DRIVE / 'captions_with_splits.csv'
CHECKPOINTS_DIR = PROJECT_DIR / 'checkpoints'
RESULTS_DIR     = PROJECT_DIR / 'results'
REMOTECLIP_CACHE = PROJECT_DIR / 'remoteclip_cache'
CHECKPOINTS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
REMOTECLIP_CACHE.mkdir(exist_ok=True)

# A local SSD copy of the image and mask data avoids the Drive I/O bottleneck
# during training.
LOCAL_DATA        = Path('/content/data')
LOCAL_IMAGES      = LOCAL_DATA / 'images'
LOCAL_MASKS_CLASS = LOCAL_DATA / 'masks_class'

if not LOCAL_DATA.exists():
    print('Copying data to local SSD...')
    shutil.copytree(DATA_DIR_DRIVE / 'images', LOCAL_IMAGES)
    shutil.copytree(DATA_DIR_DRIVE / 'masks_class', LOCAL_MASKS_CLASS)
    print('Done.')
else:
    print('Local data already present.')

print(f'Images: {LOCAL_IMAGES}')
print(f'Masks : {LOCAL_MASKS_CLASS}')

Copying data to local SSD...
Done.
Images: /content/data/images
Masks : /content/data/masks_class


In [5]:
# Reproducibility and experiment tracking
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

WANDB_PROJECT = 'di725_project'
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: selingoktas98 (selingoktas98-metu-middle-east-technical-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [6]:
# Dataset constants
CLASS_NAMES = ['Tree', 'Shrub', 'Grass', 'Crop', 'Built-up', 'Barren', 'Water']
NUM_CLASSES = len(CLASS_NAMES)

# The five caption variants available in the dataset.
CAPTION_COLS = [
    'hybrid_gemma3-4b',
    'hybrid_qwen3-vl-8b',
    'text_qwen3-4b',
    'vision_gemma3-4b',
    'vision_qwen3-vl-8b',
]

# text_qwen3-4b is the strongest text-rich variant (text-only, class-aware
# content). It is used as the fixed caption for the loss ablation and the
# additivity check.
BEST_TEXT_CAPTION = 'text_qwen3-4b'

In [7]:
# Load splits and compute class weights
split_df = pd.read_csv(SPLITS_CSV)
train_df = split_df[split_df['split'] == 'train'].reset_index(drop=True)
val_df   = split_df[split_df['split'] == 'val'].reset_index(drop=True)
test_df  = split_df[split_df['split'] == 'test'].reset_index(drop=True)
print(f'Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}')

# Median-frequency class weights (training split only) to counter class
# imbalance in the weighted cross-entropy loss.
CLASS_WEIGHT_EPS = 1e-8
class_avg    = train_df[CLASS_NAMES].mean().values
class_freq   = class_avg / class_avg.sum()
median_freq  = np.median(class_freq)
class_weights        = median_freq / (class_freq + CLASS_WEIGHT_EPS)
class_weights_tensor = torch.FloatTensor(class_weights).to(device)
print(f'Class weights: {class_weights.round(2)}')

Train: 7000  |  Val: 1500  |  Test: 1500
Class weights: [0.15 5.12 0.09 0.24 3.77 1.   2.54]


### Reference results from prior notebooks

The ablations below compare against two fixed reference points loaded from saved result files: the vision-only UNetFormer baseline from `03_baseline_models.ipynb` and the Phase 2 multimodal model (CLIP encoder with weighted cross-entropy) from `04_multimodal.ipynb`. The vision-only baseline anchors the end-to-end improvement of the full pipeline, and the multimodal baseline is the reference each Phase 3 ablation is measured against. Loading these from disk keeps the comparison consistent with the values reported in earlier phases.

In [8]:
# Reference results from prior notebooks, loaded from saved JSON files.
# Vision-only baseline: from 03_baseline_models.ipynb (baselines_results.json).
# Multimodal baseline: from 04_multimodal.ipynb (multimodal_results.json).

with open(RESULTS_DIR / 'baselines_results.json') as f:
    baseline_data = json.load(f)

BASELINE_MIOU = baseline_data['unetformer']['test_miou']
BASELINE_OA   = baseline_data['unetformer']['test_oa']
BASELINE_IOUS = baseline_data['unetformer']['class_ious']

with open(RESULTS_DIR / 'multimodal_results.json') as f:
    multimodal_data = json.load(f)

# The Phase 2 multimodal anchor is the model trained on BEST_TEXT_CAPTION
# with weighted cross-entropy and gated cross-attention.
MULTIMODAL_MIOU = multimodal_data['multimodal'][BEST_TEXT_CAPTION]['test_miou']
MULTIMODAL_OA   = multimodal_data['multimodal'][BEST_TEXT_CAPTION]['test_oa']
MULTIMODAL_IOUS = multimodal_data['multimodal'][BEST_TEXT_CAPTION]['class_ious']

print('Reference results loaded.')
print(f'  Vision-only baseline : test mIoU {BASELINE_MIOU:.4f}, OA {BASELINE_OA:.4f}')
print(f'  Multimodal baseline  : test mIoU {MULTIMODAL_MIOU:.4f}, OA {MULTIMODAL_OA:.4f}')
print(f'    (caption {BEST_TEXT_CAPTION}, weighted CE, gated cross-attention)')

Reference results loaded.
  Vision-only baseline : test mIoU 0.6488, OA 0.8621
  Multimodal baseline  : test mIoU 0.6970, OA 0.8894
    (caption text_qwen3-4b, weighted CE, gated cross-attention)


## 2. Dataset and DataLoaders

The dataset returns (image, mask, caption) tuples. The caption column is a constructor argument, so the same class serves any of the five caption variants. Masks are the pre-converted class-index masks produced in `02_preprocessing.ipynb`. The DataLoader factory builds train, validation, and test loaders for a given caption column, with shuffling on the training loader only. This matches `04_multimodal.ipynb`.

In [9]:
# Dataset class. Reads pre-converted class-index masks and one caption column.
class RSMultiModalDataset(Dataset):
    def __init__(self, dataframe, images_dir, masks_dir, caption_col):
        self.df = dataframe.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.caption_col = caption_col
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = row['filename']

        img = Image.open(self.images_dir / fname).convert('RGB')
        img = transforms.functional.to_tensor(img)
        img = self.normalize(img)

        mask = np.array(Image.open(self.masks_dir / fname))
        mask = torch.from_numpy(mask).long()

        caption = str(row[self.caption_col])
        return img, mask, caption

In [10]:
# DataLoader factory. Produces train, val, and test loaders for any caption column.
BATCH_SIZE = 8
NUM_WORKERS = 4


def make_loaders(caption_col):
    """Build train, val, and test loaders for a given caption column."""
    train_dataset = RSMultiModalDataset(train_df, LOCAL_IMAGES, LOCAL_MASKS_CLASS, caption_col)
    val_dataset = RSMultiModalDataset(val_df, LOCAL_IMAGES, LOCAL_MASKS_CLASS, caption_col)
    test_dataset = RSMultiModalDataset(test_df, LOCAL_IMAGES, LOCAL_MASKS_CLASS, caption_col)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader, test_loader

In [11]:
# Sanity check with the first caption variant
train_loader, val_loader, test_loader = make_loaders(CAPTION_COLS[0])
print(f'Caption column: {CAPTION_COLS[0]}')
print(f'Train: {len(train_loader.dataset)} samples, {len(train_loader)} batches')

imgs, masks, captions = next(iter(train_loader))
print(f'Batch: images {imgs.shape}, masks {masks.shape}')
print(f'First caption: "{captions[0][:80]}..."')

Caption column: hybrid_gemma3-4b
Train: 7000 samples, 875 batches
Batch: images torch.Size([8, 3, 256, 256]), masks torch.Size([8, 256, 256])
First caption: "The image depicts a landscape dominated by extensive grasslands (81%) interspers..."


## 3. UNetFormer architecture

The same UNetFormer used in `03_baseline_models.ipynb` and `04_multimodal.ipynb`: a `swsl_resnet18` CNN encoder, a Global-Local Transformer Block decoder, a Feature Refinement Head, and an auxiliary head for deep supervision. The architecture is re-defined here so this notebook runs standalone, and it is kept identical to the earlier phases so the ablation comparisons stay valid.

Source: https://github.com/WangLibo1995/GeoSeg

In [12]:
class ConvBNReLU(nn.Sequential):
    """Conv2d then BatchNorm then ReLU6."""
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1, stride=1,
                 norm_layer=nn.BatchNorm2d, bias=False):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, bias=bias,
                      dilation=dilation, stride=stride,
                      padding=((stride - 1) + dilation * (kernel_size - 1)) // 2),
            norm_layer(out_channels), nn.ReLU6())


class ConvBN(nn.Sequential):
    """Conv2d then BatchNorm (no activation)."""
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1, stride=1,
                 norm_layer=nn.BatchNorm2d, bias=False):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, bias=bias,
                      dilation=dilation, stride=stride,
                      padding=((stride - 1) + dilation * (kernel_size - 1)) // 2),
            norm_layer(out_channels))


class Conv(nn.Sequential):
    """Standalone Conv2d, no normalization or activation."""
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1, stride=1, bias=False):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, bias=bias,
                      dilation=dilation, stride=stride,
                      padding=((stride - 1) + dilation * (kernel_size - 1)) // 2))


class SeparableConvBN(nn.Sequential):
    """Depthwise separable convolution: depthwise then BatchNorm then pointwise (1x1).

    From the UNetFormer reference. BatchNorm uses out_channels, which is safe
    because every use in this architecture has in_channels equal to out_channels.
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, dilation=1,
                 norm_layer=nn.BatchNorm2d):
        super().__init__(
            nn.Conv2d(in_channels, in_channels, kernel_size, stride=stride, dilation=dilation,
                      padding=((stride - 1) + dilation * (kernel_size - 1)) // 2,
                      groups=in_channels, bias=False),
            norm_layer(out_channels),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False))

In [13]:
class Mlp(nn.Module):
    """Two-layer MLP using 1x1 convolutions, used inside transformer blocks."""

    def __init__(self, in_features, hidden_features=None, out_features=None,
                 act_layer=nn.ReLU6, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Conv2d(in_features, hidden_features, 1, 1, 0, bias=True)
        self.act = act_layer()
        self.fc2 = nn.Conv2d(hidden_features, out_features, 1, 1, 0, bias=True)
        self.drop = nn.Dropout(drop, inplace=True)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


class GlobalLocalAttention(nn.Module):
    """Window self-attention (global) combined with a local conv branch."""

    def __init__(self, dim=256, num_heads=16, qkv_bias=False,
                 window_size=8, relative_pos_embedding=True):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // self.num_heads
        self.scale = head_dim ** -0.5
        self.ws = window_size
        self.qkv = Conv(dim, 3 * dim, kernel_size=1, bias=qkv_bias)
        self.local1 = ConvBN(dim, dim, kernel_size=3)
        self.local2 = ConvBN(dim, dim, kernel_size=1)
        self.proj = SeparableConvBN(dim, dim, kernel_size=window_size)
        self.attn_x = nn.AvgPool2d(kernel_size=(window_size, 1), stride=1,
                                   padding=(window_size // 2 - 1, 0))
        self.attn_y = nn.AvgPool2d(kernel_size=(1, window_size), stride=1,
                                   padding=(0, window_size // 2 - 1))
        self.relative_pos_embedding = relative_pos_embedding
        if self.relative_pos_embedding:
            self.relative_position_bias_table = nn.Parameter(
                torch.zeros((2 * window_size - 1) * (2 * window_size - 1), num_heads))
            coords_h = torch.arange(self.ws)
            coords_w = torch.arange(self.ws)
            coords = torch.stack(torch.meshgrid([coords_h, coords_w]))
            coords_flatten = torch.flatten(coords, 1)
            relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
            relative_coords = relative_coords.permute(1, 2, 0).contiguous()
            relative_coords[:, :, 0] += self.ws - 1
            relative_coords[:, :, 1] += self.ws - 1
            relative_coords[:, :, 0] *= 2 * self.ws - 1
            relative_position_index = relative_coords.sum(-1)
            self.register_buffer("relative_position_index", relative_position_index)
            trunc_normal_(self.relative_position_bias_table, std=.02)

    def pad(self, x, ps):
        _, _, H, W = x.size()
        if W % ps != 0:
            x = F.pad(x, (0, ps - W % ps), mode='reflect')
        if H % ps != 0:
            x = F.pad(x, (0, 0, 0, ps - H % ps), mode='reflect')
        return x

    def pad_out(self, x):
        return F.pad(x, pad=(0, 1, 0, 1), mode='reflect')

    def forward(self, x):
        B, C, H, W = x.shape
        local = self.local2(x) + self.local1(x)
        x = self.pad(x, self.ws)
        B, C, Hp, Wp = x.shape
        qkv = self.qkv(x)
        q, k, v = rearrange(qkv, 'b (qkv h d) (hh ws1) (ww ws2) -> qkv (b hh ww) h (ws1 ws2) d',
                            h=self.num_heads, d=C // self.num_heads, hh=Hp // self.ws, ww=Wp // self.ws,
                            qkv=3, ws1=self.ws, ws2=self.ws)
        dots = (q @ k.transpose(-2, -1)) * self.scale
        if self.relative_pos_embedding:
            relative_position_bias = self.relative_position_bias_table[
                self.relative_position_index.view(-1)].view(self.ws * self.ws, self.ws * self.ws, -1)
            relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()
            dots += relative_position_bias.unsqueeze(0)
        attn = dots.softmax(dim=-1)
        attn = attn @ v
        attn = rearrange(attn, '(b hh ww) h (ws1 ws2) d -> b (h d) (hh ws1) (ww ws2)',
                         h=self.num_heads, d=C // self.num_heads, hh=Hp // self.ws, ww=Wp // self.ws,
                         ws1=self.ws, ws2=self.ws)
        attn = attn[:, :, :H, :W]
        out = self.attn_x(F.pad(attn, pad=(0, 0, 0, 1), mode='reflect')) + \
              self.attn_y(F.pad(attn, pad=(0, 1, 0, 0), mode='reflect'))
        out = out + local
        out = self.pad_out(out)
        out = self.proj(out)
        out = out[:, :, :H, :W]
        return out

In [14]:
class Block(nn.Module):
    """Global-Local Transformer Block (GLTB): pre-norm attention then MLP, both with residuals."""

    def __init__(self, dim=256, num_heads=16, mlp_ratio=4., qkv_bias=False,
                 drop=0., drop_path=0.,
                 act_layer=nn.ReLU6, norm_layer=nn.BatchNorm2d, window_size=8):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.attn = GlobalLocalAttention(dim, num_heads=num_heads, qkv_bias=qkv_bias,
                                         window_size=window_size)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, out_features=dim,
                       act_layer=act_layer, drop=drop)
        self.norm2 = norm_layer(dim)

    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class WF(nn.Module):
    """Weighted feature fusion: upsamples low-resolution features and combines them
    with skip-connection features using learned, ReLU-normalized weights."""

    def __init__(self, in_channels=128, decode_channels=128, eps=1e-8):
        super().__init__()
        self.pre_conv = Conv(in_channels, decode_channels, kernel_size=1)
        self.weights = nn.Parameter(torch.ones(2, dtype=torch.float32), requires_grad=True)
        self.eps = eps
        self.post_conv = ConvBNReLU(decode_channels, decode_channels, kernel_size=3)

    def forward(self, x, res):
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        weights = nn.ReLU()(self.weights)
        fuse_weights = weights / (torch.sum(weights, dim=0) + self.eps)
        x = fuse_weights[0] * self.pre_conv(res) + fuse_weights[1] * x
        return self.post_conv(x)


class FeatureRefinementHead(nn.Module):
    """Final decoder stage: weighted fusion, spatial and channel attention, residual projection."""

    def __init__(self, in_channels=64, decode_channels=64):
        super().__init__()
        self.pre_conv = Conv(in_channels, decode_channels, kernel_size=1)
        self.weights = nn.Parameter(torch.ones(2, dtype=torch.float32), requires_grad=True)
        self.eps = 1e-8
        self.post_conv = ConvBNReLU(decode_channels, decode_channels, kernel_size=3)
        self.pa = nn.Sequential(
            nn.Conv2d(decode_channels, decode_channels, kernel_size=3, padding=1,
                      groups=decode_channels), nn.Sigmoid())
        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            Conv(decode_channels, decode_channels // 16, kernel_size=1), nn.ReLU6(),
            Conv(decode_channels // 16, decode_channels, kernel_size=1), nn.Sigmoid())
        self.shortcut = ConvBN(decode_channels, decode_channels, kernel_size=1)
        self.proj = SeparableConvBN(decode_channels, decode_channels, kernel_size=3)
        self.act = nn.ReLU6()

    def forward(self, x, res):
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        weights = nn.ReLU()(self.weights)
        fuse_weights = weights / (torch.sum(weights, dim=0) + self.eps)
        x = fuse_weights[0] * self.pre_conv(res) + fuse_weights[1] * x
        x = self.post_conv(x)
        shortcut = self.shortcut(x)
        pa = self.pa(x) * x
        ca = self.ca(x) * x
        x = pa + ca
        x = self.proj(x) + shortcut
        return self.act(x)


class AuxHead(nn.Module):
    """Auxiliary segmentation head used during training for deep supervision."""

    def __init__(self, in_channels=64, num_classes=8):
        super().__init__()
        self.conv = ConvBNReLU(in_channels, in_channels)
        self.drop = nn.Dropout(0.1)
        self.conv_out = Conv(in_channels, num_classes, kernel_size=1)

    def forward(self, x, h, w):
        feat = self.conv(x)
        feat = self.drop(feat)
        feat = self.conv_out(feat)
        return F.interpolate(feat, size=(h, w), mode='bilinear', align_corners=False)

## 4. Multimodal architecture

The multimodal model extends UNetFormer with two additions:

1. **Frozen CLIP text encoder** (`ViT-B-32`, `laion2b_s34b_b79k`): converts each caption into a 512-d L2-normalized embedding. No gradients flow into CLIP.
2. **Gated cross-attention**: visual features at each decoder stage are conditioned on the text embedding through a gated residual. The gate starts at zero, so text has no influence at the start of training and the model initially behaves like the vision-only baseline. The gate then learns how much text to incorporate.

The CNN backbone (`swsl_resnet18`) is initialized from ImageNet-pretrained weights and fine-tuned together with the decoder and the cross-attention modules, while the CLIP text encoder stays frozen. This setup is identical to `04_multimodal.ipynb` so that all Phase 3 comparisons are consistent.

In [15]:
class CLIPTextEncoder(nn.Module):
    """Frozen CLIP text encoder. Outputs L2-normalized 512-d embeddings."""

    def __init__(self, model_name='ViT-B-32', pretrained='laion2b_s34b_b79k'):
        super().__init__()
        clip_model, _, _ = open_clip.create_model_and_transforms(model_name, pretrained=pretrained)
        self.clip_model = clip_model
        self.tokenizer = open_clip.get_tokenizer(model_name)

        # Drop the visual tower since only the text encoder is used.
        # This saves about 88M frozen parameters of GPU memory.
        if hasattr(self.clip_model, 'visual'):
            del self.clip_model.visual

        # Freeze all remaining CLIP parameters.
        for param in self.parameters():
            param.requires_grad = False

    @torch.no_grad()
    def forward(self, text_list):
        device = next(self.parameters()).device
        tokens = self.tokenizer(text_list).to(device)
        text_features = self.clip_model.encode_text(tokens)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        return text_features


# Verify the encoder loads and the visual tower is dropped.
clip_enc = CLIPTextEncoder().to(device)
with torch.no_grad():
    emb = clip_enc(['test caption'])
    print(f'CLIP text encoder loaded: output {emb.shape}')
    print(f'Frozen params: {sum(p.numel() for p in clip_enc.parameters()) / 1e6:.1f}M')
del clip_enc
torch.cuda.empty_cache()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

CLIP text encoder loaded: output torch.Size([1, 512])
Frozen params: 63.4M


In [16]:
class TextVisualCrossAttention(nn.Module):
    """Conditions visual features on the caption embedding through a gated residual.

    The caption is provided as a single pooled CLIP embedding. The module applies
    query, key, and value projections and a gated residual. The gate starts at 0,
    so text has no influence at the start of training and the model initially
    behaves like the vision-only baseline. The gate then learns how much text to
    incorporate.
    """

    def __init__(self, visual_dim=64, text_dim=512, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = visual_dim // num_heads
        self.scale = self.head_dim ** -0.5

        # Project the text embedding into the visual feature space.
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, visual_dim), nn.ReLU6(),
            nn.Linear(visual_dim, visual_dim),
        )

        self.q_proj = nn.Conv2d(visual_dim, visual_dim, 1)
        self.k_proj = nn.Linear(visual_dim, visual_dim)
        self.v_proj = nn.Linear(visual_dim, visual_dim)
        self.out_proj = nn.Sequential(
            nn.Conv2d(visual_dim, visual_dim, 1), nn.BatchNorm2d(visual_dim),
        )

        # Gated residual. Starts at 0 and learns to scale the text contribution.
        self.gate = nn.Parameter(torch.zeros(1))

    def forward(self, visual, text_embed):
        B, C, H, W = visual.shape
        text_feat = self.text_proj(text_embed)
        q = self.q_proj(visual).reshape(B, self.num_heads, self.head_dim, H * W).permute(0, 1, 3, 2)
        k = self.k_proj(text_feat).reshape(B, self.num_heads, 1, self.head_dim)
        v = self.v_proj(text_feat).reshape(B, self.num_heads, 1, self.head_dim)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        out = (attn @ v).permute(0, 1, 3, 2).reshape(B, C, H, W)
        out = self.out_proj(out)
        return visual + self.gate * out

In [17]:
class TextAugmentedDecoder(nn.Module):
    """UNetFormer decoder with gated cross-attention after each decoder stage."""

    def __init__(self, encoder_channels=(64, 128, 256, 512), decode_channels=64,
                 dropout=0.1, window_size=8, num_classes=7, text_dim=512):
        super().__init__()
        self.pre_conv = ConvBN(encoder_channels[-1], decode_channels, kernel_size=1)
        self.b4 = Block(dim=decode_channels, num_heads=8, window_size=window_size)
        self.p3 = WF(encoder_channels[-2], decode_channels)
        self.b3 = Block(dim=decode_channels, num_heads=8, window_size=window_size)
        self.p2 = WF(encoder_channels[-3], decode_channels)
        self.b2 = Block(dim=decode_channels, num_heads=8, window_size=window_size)
        self.p1 = FeatureRefinementHead(encoder_channels[-4], decode_channels)
        self.up4 = nn.UpsamplingBilinear2d(scale_factor=4)
        self.up3 = nn.UpsamplingBilinear2d(scale_factor=2)
        self.aux_head = AuxHead(decode_channels, num_classes)
        self.segmentation_head = nn.Sequential(
            ConvBNReLU(decode_channels, decode_channels),
            nn.Dropout2d(p=dropout, inplace=True),
            Conv(decode_channels, num_classes, kernel_size=1),
        )

        # Gated cross-attention at each decoder stage.
        self.text_attn4 = TextVisualCrossAttention(decode_channels, text_dim, num_heads=4)
        self.text_attn3 = TextVisualCrossAttention(decode_channels, text_dim, num_heads=4)
        self.text_attn2 = TextVisualCrossAttention(decode_channels, text_dim, num_heads=4)
        self.text_attn1 = TextVisualCrossAttention(decode_channels, text_dim, num_heads=4)

        self.init_weight()

    def forward(self, res1, res2, res3, res4, h, w, text_embed):
        if self.training:
            x = self.b4(self.pre_conv(res4))
            x = self.text_attn4(x, text_embed)
            h4 = self.up4(x)
            x = self.p3(x, res3)
            x = self.b3(x)
            x = self.text_attn3(x, text_embed)
            h3 = self.up3(x)
            x = self.p2(x, res2)
            x = self.b2(x)
            x = self.text_attn2(x, text_embed)
            h2 = x
            x = self.p1(x, res1)
            x = self.text_attn1(x, text_embed)
            x = self.segmentation_head(x)
            x = F.interpolate(x, size=(h, w), mode='bilinear', align_corners=False)
            ah = h4 + h3 + h2
            ah = self.aux_head(ah, h, w)
            return x, ah
        else:
            x = self.b4(self.pre_conv(res4))
            x = self.text_attn4(x, text_embed)
            x = self.p3(x, res3)
            x = self.b3(x)
            x = self.text_attn3(x, text_embed)
            x = self.p2(x, res2)
            x = self.b2(x)
            x = self.text_attn2(x, text_embed)
            x = self.p1(x, res1)
            x = self.text_attn1(x, text_embed)
            x = self.segmentation_head(x)
            return F.interpolate(x, size=(h, w), mode='bilinear', align_corners=False)

    def init_weight(self):
        for m in self.children():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, a=1)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

In [18]:
class UNetFormerCLIP(nn.Module):
    """UNetFormer with a frozen CLIP text encoder and gated cross-attention fusion."""

    def __init__(self, decode_channels=64, dropout=0.1, backbone_name='swsl_resnet18',
                 pretrained=True, window_size=8, num_classes=7,
                 clip_model='ViT-B-32', clip_pretrained='laion2b_s34b_b79k'):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, features_only=True, output_stride=32,
                                          out_indices=(1, 2, 3, 4), pretrained=pretrained)
        encoder_channels = self.backbone.feature_info.channels()
        self.text_encoder = CLIPTextEncoder(clip_model, clip_pretrained)
        self.decoder = TextAugmentedDecoder(encoder_channels, decode_channels, dropout,
                                            window_size, num_classes, text_dim=512)

    def forward(self, x, captions):
        h, w = x.size()[-2:]
        res1, res2, res3, res4 = self.backbone(x)
        text_embed = self.text_encoder(captions)
        if self.training:
            out, ah = self.decoder(res1, res2, res3, res4, h, w, text_embed)
            return out, ah
        else:
            return self.decoder(res1, res2, res3, res4, h, w, text_embed)

In [19]:
# Sanity check: build the model and run one forward pass in train and eval mode.
sample_model = UNetFormerCLIP(num_classes=NUM_CLASSES).to(device)

n_total = sum(p.numel() for p in sample_model.parameters())
n_train = sum(p.numel() for p in sample_model.parameters() if p.requires_grad)
print(f'Total params    : {n_total / 1e6:.1f}M')
print(f'Trainable params: {n_train / 1e6:.1f}M  (backbone + decoder + cross-attention)')
print(f'Frozen params   : {(n_total - n_train) / 1e6:.1f}M  (CLIP text encoder)')

sample_imgs = torch.randn(2, 3, 256, 256).to(device)
sample_caps = ['a sample caption', 'another caption']

sample_model.train()
out_train, aux_train = sample_model(sample_imgs, sample_caps)
print(f'Train mode : main {tuple(out_train.shape)}, aux {tuple(aux_train.shape)}')

sample_model.eval()
with torch.no_grad():
    out_eval = sample_model(sample_imgs, sample_caps)
print(f'Eval mode  : {tuple(out_eval.shape)}')

del sample_model
torch.cuda.empty_cache()

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name swsl_resnet18 to current resnet18.fb_swsl_ig1b_ft_in1k.
  model = create_fn(


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Total params    : 75.4M
Trainable params: 11.9M  (backbone + decoder + cross-attention)
Frozen params   : 63.4M  (CLIP text encoder)
Train mode : main (2, 7, 256, 256), aux (2, 7, 256, 256)
Eval mode  : (2, 7, 256, 256)


## 5. Training infrastructure

Same training recipe as `04_multimodal.ipynb`: 30 epochs, AdamW (lr 6e-4, weight decay 0.01), CosineAnnealingLR (eta_min 1e-6), and weighted cross-entropy plus 0.4 times an auxiliary head loss.

The `train_multimodal` function accepts optional `model_factory` and `criterion` arguments. The defaults reproduce the Phase 2 configuration (`UNetFormerCLIP` with weighted cross-entropy). The ablations supply a different model (RemoteCLIP) and different losses (CE+Dice, CE+Lovász, CE+Tversky) without modifying the training loop.

Each model is checkpointed at its best validation mIoU and the function returns the full per-epoch validation history. Configuration selection in each ablation uses this validation mIoU. Each run is logged to Weights and Biases under the project `di725_project`.

In [20]:
# Training hyperparameters (same as 04_multimodal.ipynb)
NUM_EPOCHS      = 30
LR              = 6e-4
WEIGHT_DECAY    = 0.01
AUX_LOSS_WEIGHT = 0.4
SCHEDULER_ETA_MIN = 1e-6

In [21]:
def update_confusion(intersection, union, pred, target, num_classes=NUM_CLASSES):
    """Accumulate per-class intersection and union counts in place.

    pred and target are tensors of shape (B, H, W) with class indices.
    Computation stays on GPU and only scalars transfer to CPU via .item().
    """
    for c in range(num_classes):
        pred_c = (pred == c)
        target_c = (target == c)
        intersection[c] += (pred_c & target_c).sum().item()
        union[c] += (pred_c | target_c).sum().item()


def finalize_iou(intersection, union, num_classes=NUM_CLASSES):
    """Convert accumulated counts into per-class IoU and mIoU.

    Classes absent from both prediction and target (union 0) are NaN and
    excluded from the mIoU average.
    """
    class_ious = []
    for c in range(num_classes):
        if union[c] == 0:
            class_ious.append(float('nan'))
        else:
            class_ious.append(intersection[c] / union[c])
    valid = [x for x in class_ious if not np.isnan(x)]
    miou = np.mean(valid) if valid else 0.0
    return class_ious, miou

In [22]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    """Run one training epoch over the (image, mask, caption) loader.

    The criterion takes (logits, target_mask) and returns a scalar loss. The same
    loop is used for weighted CE and for the composite losses (CE+Dice, CE+Lovász,
    CE+Tversky).
    """
    model.train()
    total_loss = 0
    for imgs, masks, captions in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits_main, logits_aux = model(imgs, list(captions))
        loss = criterion(logits_main, masks) + AUX_LOSS_WEIGHT * criterion(logits_aux, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, criterion, device, num_classes=NUM_CLASSES):
    """Return per-class pixel IoU, pixel-level mIoU, overall accuracy, and average loss."""
    model.eval()
    intersection = np.zeros(num_classes, dtype=np.int64)
    union = np.zeros(num_classes, dtype=np.int64)
    total_correct = 0
    total_pixels = 0
    total_loss = 0
    for imgs, masks, captions in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits = model(imgs, list(captions))
        total_loss += criterion(logits, masks).item()
        preds = logits.argmax(dim=1)
        update_confusion(intersection, union, preds, masks, num_classes)
        total_correct += (preds == masks).sum().item()
        total_pixels += masks.numel()
    class_ious, miou = finalize_iou(intersection, union, num_classes)
    oa = total_correct / total_pixels
    return class_ious, miou, oa, total_loss / len(loader)

In [23]:
def train_multimodal(caption_col, model_factory=None, criterion=None,
                     num_epochs=NUM_EPOCHS, lr=LR, run_name=None, wandb_config=None):
    """Train a multimodal model and return (history, best_val_miou, checkpoint_path).

    The model is checkpointed whenever validation mIoU improves, so the saved
    checkpoint corresponds to the best validation epoch. The returned best_val_miou
    is the validation metric used for configuration selection in the ablations.

    Args:
        caption_col:   caption column from CAPTION_COLS to use as text input.
        model_factory: callable returning a fresh model. Defaults to the Phase 2
                       model (UNetFormerCLIP with gated cross-attention).
        criterion:     loss function (logits, target) returning a scalar. Defaults
                       to weighted cross-entropy.
        run_name:      identifier for the W&B run and the checkpoint filename.
        wandb_config:  dict of experiment-specific fields added to the W&B config.
    """
    # Defaults reproduce the Phase 2 configuration.
    if model_factory is None:
        model_factory = lambda: UNetFormerCLIP(num_classes=NUM_CLASSES)
    if criterion is None:
        criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    if run_name is None:
        base_name = caption_col.replace('-', '_').replace('/', '_')
        run_name = f'multimodal_{base_name}'

    save_path = CHECKPOINTS_DIR / f'{run_name}_best.pth'

    train_loader, val_loader, _ = make_loaders(caption_col)
    model = model_factory().to(device)

    # Optimize only trainable parameters. This skips the frozen CLIP text encoder
    # and includes the fine-tuned CNN backbone, the decoder, and cross-attention.
    trainable = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = AdamW(trainable, lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=SCHEDULER_ETA_MIN)

    default_config = {
        'caption_col':     caption_col,
        'num_epochs':      num_epochs,
        'lr':              lr,
        'weight_decay':    WEIGHT_DECAY,
        'batch_size':      BATCH_SIZE,
        'aux_loss_weight': AUX_LOSS_WEIGHT,
        'seed':            SEED,
    }
    if wandb_config:
        default_config.update(wandb_config)

    wandb.init(project=WANDB_PROJECT, name=run_name, config=default_config, reinit=True)

    history = {'train_loss': [], 'val_loss': [], 'val_miou': [], 'val_oa': []}
    best_val_miou = 0.0

    print(f'Training {run_name} for {num_epochs} epochs...')
    print(f"{'Epoch':>5} {'T_Loss':>8} {'V_Loss':>8} {'V_mIoU':>8} {'V_OA':>8} {'Time':>7} {'':>6}")
    print('-' * 55)

    for epoch in range(num_epochs):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_class_ious, val_miou, val_oa, val_loss = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_miou'].append(val_miou)
        history['val_oa'].append(val_oa)

        log_dict = {
            'epoch':      epoch + 1,
            'train_loss': train_loss,
            'val_loss':   val_loss,
            'val_miou':   val_miou,
            'val_oa':     val_oa,
            'lr':         scheduler.get_last_lr()[0],
        }
        for c, name in enumerate(CLASS_NAMES):
            if not np.isnan(val_class_ious[c]):
                log_dict[f'val_iou/{name}'] = val_class_ious[c]
        wandb.log(log_dict)

        # Checkpoint on best validation mIoU.
        status = ''
        if val_miou > best_val_miou:
            best_val_miou = val_miou
            torch.save(model.state_dict(), str(save_path))
            status = 'BEST'

        elapsed = time.time() - t0
        print(f'{epoch+1:>5} {train_loss:>8.4f} {val_loss:>8.4f} '
              f'{val_miou:>8.4f} {val_oa:>8.4f} {elapsed:>6.1f}s {status:>6}')

    wandb.summary['best_val_miou'] = best_val_miou
    wandb.finish()

    del model
    torch.cuda.empty_cache()

    print(f'\nBest validation mIoU: {best_val_miou:.4f}')
    return history, best_val_miou, save_path

In [24]:
def save_history(history, name):
    """Persist training history to disk so it survives runtime restarts."""
    path = RESULTS_DIR / f'{name}_history.json'
    with open(path, 'w') as f:
        json.dump(history, f)
    print(f'Saved history to {path}')


def evaluate_checkpoint(caption_col, ckpt_path, model_factory=None, criterion=None):
    """Reload a trained checkpoint and evaluate it once on the test set.

    Returns per-class IoU, test mIoU, test overall accuracy, and average test loss.
    Test evaluation is run only for the configuration selected on validation mIoU,
    and for the full ablation comparison tables reported as analysis.
    """
    if model_factory is None:
        model_factory = lambda: UNetFormerCLIP(num_classes=NUM_CLASSES)
    if criterion is None:
        criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

    _, _, test_loader = make_loaders(caption_col)
    model = model_factory().to(device)
    model.load_state_dict(torch.load(str(ckpt_path)))
    class_ious, miou, oa, loss = evaluate(model, test_loader, criterion, device)
    del model
    torch.cuda.empty_cache()
    return class_ious, miou, oa, loss

## 6. Ablation 1: Text encoder

This ablation tests whether replacing CLIP with RemoteCLIP changes segmentation performance. CLIP (`ViT-B-32`, `laion2b_s34b_b79k`) is pretrained on generic web image and text pairs. RemoteCLIP (Liu et al., 2023) is a CLIP variant fine-tuned on remote sensing image and text datasets (RSITMD, RSICD, Sydney-Captions). Both share the same `ViT-B-32` architecture, so only the encoder weights change.

All five caption variants are trained so the encoder swap can be tested per caption. Each variant is checkpointed on validation mIoU. The best caption variant is selected by validation mIoU, and that selected configuration is the one carried into the additivity check. Per-caption test mIoU is reported afterwards as ablation analysis. Phase 2 CLIP test results are loaded from `multimodal_results.json` for comparison.

In [25]:
class CLIPTextEncoderRC(nn.Module):
    """Frozen text encoder that supports either CLIP or RemoteCLIP weights.

    Both use the ViT-B-32 architecture and only the trained weights differ.
    RemoteCLIP weights are downloaded from HuggingFace (chendelong/RemoteCLIP)
    and loaded into an open_clip ViT-B-32 model created with no pretrained weights.
    """

    def __init__(self, model_name='ViT-B-32', pretrained='laion2b_s34b_b79k'):
        super().__init__()
        if pretrained == 'remoteclip':
            # Create an empty ViT-B-32, then load RemoteCLIP weights into it.
            clip_model, _, _ = open_clip.create_model_and_transforms(model_name, pretrained=None)
            ckpt_path = hf_hub_download(
                'chendelong/RemoteCLIP',
                f'RemoteCLIP-{model_name}.pt',
                cache_dir=str(REMOTECLIP_CACHE),
            )
            ckpt = torch.load(ckpt_path, map_location='cpu')
            msg = clip_model.load_state_dict(ckpt)
            print(f'RemoteCLIP {model_name} loaded: {msg}')
        else:
            clip_model, _, _ = open_clip.create_model_and_transforms(model_name, pretrained=pretrained)

        self.clip_model = clip_model
        self.tokenizer = open_clip.get_tokenizer(model_name)

        if hasattr(self.clip_model, 'visual'):
            del self.clip_model.visual

        for param in self.parameters():
            param.requires_grad = False

    @torch.no_grad()
    def forward(self, text_list):
        device = next(self.parameters()).device
        tokens = self.tokenizer(text_list).to(device)
        text_features = self.clip_model.encode_text(tokens)
        return text_features / text_features.norm(dim=-1, keepdim=True)


class UNetFormerRemoteCLIP(nn.Module):
    """UNetFormer with a frozen RemoteCLIP text encoder and gated cross-attention.

    The architecture is identical to UNetFormerCLIP except for the text encoder
    weights, so all comparisons isolate the effect of the encoder.
    """

    def __init__(self, decode_channels=64, dropout=0.1, backbone_name='swsl_resnet18',
                 pretrained=True, window_size=8, num_classes=7):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, features_only=True, output_stride=32,
                                          out_indices=(1, 2, 3, 4), pretrained=pretrained)
        encoder_channels = self.backbone.feature_info.channels()
        self.text_encoder = CLIPTextEncoderRC(pretrained='remoteclip')
        self.decoder = TextAugmentedDecoder(encoder_channels, decode_channels, dropout,
                                            window_size, num_classes, text_dim=512)

    def forward(self, x, captions):
        h, w = x.size()[-2:]
        res1, res2, res3, res4 = self.backbone(x)
        text_embed = self.text_encoder(captions)
        if self.training:
            out, ah = self.decoder(res1, res2, res3, res4, h, w, text_embed)
            return out, ah
        else:
            return self.decoder(res1, res2, res3, res4, h, w, text_embed)

In [26]:
# Sanity check: the RemoteCLIP model builds, the encoder swap works, and the
# trainable parameter count matches the CLIP-based model.
test_model = UNetFormerRemoteCLIP(num_classes=NUM_CLASSES).to(device)
total = sum(p.numel() for p in test_model.parameters()) / 1e6
trainable = sum(p.numel() for p in test_model.parameters() if p.requires_grad) / 1e6
print(f'Total parameters    : {total:.2f}M')
print(f'Trainable parameters: {trainable:.2f}M (same as the CLIP-based model)')

with torch.no_grad():
    sample_imgs, _, sample_caps = next(iter(train_loader))
    sample_imgs = sample_imgs[:2].to(device)
    sample_caps = list(sample_caps[:2])
    test_model.eval()
    out = test_model(sample_imgs, sample_caps)
    print(f'Forward pass: {tuple(sample_imgs.shape)} -> {tuple(out.shape)}')

del test_model
torch.cuda.empty_cache()

RemoteCLIP ViT-B-32 loaded: <All keys matched successfully>
Total parameters    : 75.37M
Trainable parameters: 11.94M (same as the CLIP-based model)
Forward pass: (2, 3, 256, 256) -> (2, 7, 256, 256)


In [ ]:
# Train UNetFormerRemoteCLIP on all five caption variants.
# Each run is checkpointed on best validation mIoU and returns its validation history.
remoteclip_factory = lambda: UNetFormerRemoteCLIP(num_classes=NUM_CLASSES)
remoteclip_runs = {}

for caption_col in CAPTION_COLS:
    base_name = caption_col.replace('-', '_').replace('/', '_')
    run_name = f'remoteclip_{base_name}'

    print(f'\n{"=" * 60}')
    print(f'Caption: {caption_col}')
    print(f'{"=" * 60}\n')

    hist, val_miou, ckpt = train_multimodal(
        caption_col=caption_col,
        model_factory=remoteclip_factory,
        run_name=run_name,
        wandb_config={'text_encoder': 'remoteclip', 'experiment': 'ablation_1_encoder'},
    )
    save_history(hist, run_name)
    remoteclip_runs[caption_col] = {'history': hist, 'val_miou': val_miou, 'ckpt': ckpt}


Caption: hybrid_gemma3-4b



RemoteCLIP ViT-B-32 loaded: <All keys matched successfully>


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Training remoteclip_hybrid_gemma3_4b for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   1.1056   0.5380   0.5201   0.7252   57.9s   BEST
    2   0.7246   0.4827   0.4969   0.7446   56.7s       
    3   0.6155   0.5638   0.4375   0.7184   56.0s       
    4   0.5672   0.3895   0.5764   0.7822   57.1s   BEST
    5   0.5172   0.3616   0.5951   0.7933   57.8s   BEST
    6   0.5023   0.3792   0.5913   0.7859   57.6s       
    7   0.4743   0.3315   0.5735   0.7981   56.5s       
    8   0.4542   0.3366   0.6010   0.8069   58.4s   BEST
    9   0.4330   0.2719   0.6088   0.8237   57.3s   BEST
   10   0.4115   0.2792   0.6120   0.8212   57.7s   BEST
   11   0.3990   0.2862   0.6172   0.8294   57.1s   BEST
   12   0.3769   0.2567   0.6434   0.8562   57.4s   BEST
   13   0.3785   0.2738   0.6240   0.8510   56.4s       
   14   0.3623   0.2581   0.6357   0.8371   57.1s       
   15   0.3624   0.2502   0.6468   

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▂▄▁▂▄▄▃▅▄▃▄▇▆▅▇▅▆▇▅███▇███▇▇██
val_iou/Built-up,▅▁▅▇▇█▅▆▇▇▆▇▄▇▆▇▆▇▇▇▇█▇█▇▇███▇
val_iou/Crop,▂▁▁▃▄▄▄▅▅▅▆▆▆▇▇▇▇▇▇██▇████████
val_iou/Grass,▁▃▃▄▄▄▅▆▆▅▆▇▇▆▇▆▇▇▇▇██████████
val_iou/Shrub,▁▂▃▃▁▁▃▁▃▃▄▅▄▃▄▄▆▇▆▅▇▆▆▇▇▇▇█▇▇
val_iou/Tree,▃▃▁▅▆▅▅▅▇▇▆▇▇▇▇▇▇██▇██████████
val_iou/Water,▇▆▁▇▇▇▇█▇▇█▇▇████▇███▇████████
+3,...


...


Best validation mIoU: 0.6502
Saved history to /content/drive/MyDrive/DI725_Project/results/remoteclip_vision_gemma3_4b_history.json

Caption: vision_qwen3-vl-8b



RemoteCLIP ViT-B-32 loaded: <All keys matched successfully>


Training remoteclip_vision_qwen3_vl_8b for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   1.3189   1.0414   0.3599   0.5804   57.9s   BEST
    2   0.9942   0.7167   0.3901   0.6827   59.0s   BEST
    3   0.9039   0.7800   0.4514   0.6108   58.2s   BEST
    4   0.8286   0.5120   0.4824   0.7072   57.5s   BEST
    5   0.8112   0.6190   0.4447   0.6974   57.8s       
    6   0.7181   0.4789   0.5200   0.7398   58.7s   BEST
    7   0.7323   0.4092   0.5393   0.7590   58.4s   BEST
    8   0.6686   0.4151   0.5832   0.8008   57.6s   BEST
    9   0.6596   0.4665   0.4805   0.7377   57.5s       
   10   0.6063   0.5083   0.4907   0.7117   57.9s       
   11   0.5961   0.3701   0.5706   0.7885   56.8s       
   12   0.5719   0.4229   0.5493   0.7775   57.2s       
   13   0.5445   0.4279   0.5816   0.7851   57.3s       
   14   0.5199   0.3481   0.5982   0.8261   58.1s   BEST
   15   0.4900   0.3959   0.5861 

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▆▅▅▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▁▁▂▁▄▁▃▇▄▃▅▅▅▆▇▅▅▇▆▅▇▇▇▇▇▆██▇▇
val_iou/Built-up,▄▃▃▄▅▅█▇▁▅▆▆█▆▅▄▅▄█▇▇▇▇██▇██▇█
val_iou/Crop,▁▃▄▂▃▄▅▆▃▃▅▆▄▇▆▆▇▇▇▇▇█████████
val_iou/Grass,▁▄▁▄▂▅▅▆▄▅▆▅▆▇▇▆▇▇▇▇▇▇███▇████
val_iou/Shrub,▁▃▁▄▂▃▄▄▆▃▄▅▆▆▅▆▄▇▇▆▄▆██▇▇████
val_iou/Tree,▁▄▄▅▅▅▆▇▆▂▆▆▇▇▆▇▇▇▇▇▇▇████████
val_iou/Water,▂▁▇▆▄▇▆▇▅▆█▇█▇█▆███▇▇█████████
+3,...



Best validation mIoU: 0.6507
Saved history to /content/drive/MyDrive/DI725_Project/results/remoteclip_vision_qwen3_vl_8b_history.json


In [28]:
# Select the best caption variant by VALIDATION mIoU.
# Selection never uses the test set. remoteclip_runs[c]['val_miou'] is the best
# validation mIoU returned by train_multimodal (the checkpoint-selection metric).
best_rc_caption = max(remoteclip_runs, key=lambda c: remoteclip_runs[c]['val_miou'])
best_rc_val_miou = remoteclip_runs[best_rc_caption]['val_miou']

print('Validation mIoU per caption (RemoteCLIP):')
print(f"{'Caption':<22} {'val mIoU':>10}")
print('-' * 34)
for caption_col in CAPTION_COLS:
    marker = '  <- selected' if caption_col == best_rc_caption else ''
    print(f'{caption_col:<22} {remoteclip_runs[caption_col]["val_miou"]:>10.4f}{marker}')

print(f'\nSelected caption (by validation mIoU): {best_rc_caption}')
print(f'Validation mIoU: {best_rc_val_miou:.4f}')

Validation mIoU per caption (RemoteCLIP):
Caption                  val mIoU
----------------------------------
hybrid_gemma3-4b           0.6996
hybrid_qwen3-vl-8b         0.6974
text_qwen3-4b              0.7014  <- selected
vision_gemma3-4b           0.6502
vision_qwen3-vl-8b         0.6507

Selected caption (by validation mIoU): text_qwen3-4b
Validation mIoU: 0.7014


**Observation:** On validation mIoU, the text and hybrid captions cluster at the top (`text_qwen3-4b`, `hybrid_gemma3-4b`, `hybrid_qwen3-vl-8b`), and the two vision captions are clearly lower. `text_qwen3-4b` is the highest and is selected as the caption carried forward, matching the best caption from Phase 2. The stability of the best caption across encoders suggests the caption ranking is driven by caption content rather than the choice of text encoder.

In [29]:
# Test-set evaluation for all five RemoteCLIP checkpoints.
# This loop only computes and stores results; the formatted table is printed
# in the next cell. Configuration selection used validation mIoU only.
remoteclip_test = {}
print('Evaluating RemoteCLIP checkpoints on the test set...')

for caption_col, run_info in remoteclip_runs.items():
    class_ious, miou, oa, loss = evaluate_checkpoint(
        caption_col=caption_col,
        ckpt_path=run_info['ckpt'],
        model_factory=remoteclip_factory,
    )
    remoteclip_test[caption_col] = {
        'class_ious': class_ious, 'miou': miou, 'oa': oa, 'loss': loss,
    }

# Test mIoU of the validation-selected caption, reported as the headline value.
best_rc_test_miou = remoteclip_test[best_rc_caption]['miou']
best_rc_test_oa   = remoteclip_test[best_rc_caption]['oa']

Evaluating RemoteCLIP checkpoints on the test set...


RemoteCLIP ViT-B-32 loaded: <All keys matched successfully>


RemoteCLIP ViT-B-32 loaded: <All keys matched successfully>


RemoteCLIP ViT-B-32 loaded: <All keys matched successfully>


RemoteCLIP ViT-B-32 loaded: <All keys matched successfully>


RemoteCLIP ViT-B-32 loaded: <All keys matched successfully>


In [30]:
# Clean summary of the RemoteCLIP test results.
# best_rc_caption was selected on validation in the selection cell above.
print('RemoteCLIP test results per caption:\n')
print(f"{'Caption':<22}{'test mIoU':>12}{'test OA':>10}")
print('-' * 44)
for caption, m in remoteclip_test.items():
    marker = '  <- selected' if caption == best_rc_caption else ''
    print(f"{caption:<22}{m['miou']:>12.4f}{m['oa']:>10.4f}{marker}")

print(f"\nSelected caption (by validation mIoU): {best_rc_caption}")
print(f"Test mIoU {best_rc_test_miou:.4f}, test OA {best_rc_test_oa:.4f}")

RemoteCLIP test results per caption:

Caption                  test mIoU   test OA
--------------------------------------------
hybrid_gemma3-4b            0.6960    0.8878
hybrid_qwen3-vl-8b          0.6923    0.8870
text_qwen3-4b               0.7008    0.8904  <- selected
vision_gemma3-4b            0.6450    0.8548
vision_qwen3-vl-8b          0.6479    0.8588

Selected caption (by validation mIoU): text_qwen3-4b
Test mIoU 0.7008, test OA 0.8904


In [31]:
# Paired comparison: Phase 2 CLIP vs Phase 3 RemoteCLIP, per caption (analysis).
phase2_clip = multimodal_data['multimodal']

print(f"{'Caption':<22} {'CLIP':>10} {'RemoteCLIP':>12} {'Delta':>10} {'Rel %':>8}")
print('-' * 64)

for caption_col in CAPTION_COLS:
    clip_miou = phase2_clip[caption_col]['test_miou']
    rc_miou = remoteclip_test[caption_col]['miou']
    delta = rc_miou - clip_miou
    rel = (delta / clip_miou) * 100
    print(f'{caption_col:<22} {clip_miou:>10.4f} {rc_miou:>12.4f} {delta:>+10.4f} {rel:>+7.2f}%')

Caption                      CLIP   RemoteCLIP      Delta    Rel %
----------------------------------------------------------------
hybrid_gemma3-4b           0.6930       0.6960    +0.0030   +0.44%
hybrid_qwen3-vl-8b         0.6906       0.6923    +0.0018   +0.26%
text_qwen3-4b              0.6970       0.7008    +0.0038   +0.55%
vision_gemma3-4b           0.6393       0.6450    +0.0057   +0.89%
vision_qwen3-vl-8b         0.6519       0.6479    -0.0040   -0.61%


**Observation:** On the test set, RemoteCLIP improves over CLIP on the two hybrid captions, the text caption, and `vision_gemma3-4b`, and regresses only on `vision_qwen3-vl-8b`. The gains are small, between +0.0018 and +0.0057, and on the selected caption `text_qwen3-4b` RemoteCLIP is ahead by +0.0038. The effect of the encoder swap is consistent in direction but modest in size on this dataset.

In [32]:
# Per-class IoU comparison on the validation-selected caption: CLIP vs RemoteCLIP.
clip_ious_on_selected = phase2_clip[best_rc_caption]['class_ious']
rc_ious_on_selected = remoteclip_test[best_rc_caption]['class_ious']
clip_miou_on_selected = phase2_clip[best_rc_caption]['test_miou']

print(f'Per-class comparison on {best_rc_caption} (CLIP vs RemoteCLIP):\n')
print(f"{'Class':<12} {'CLIP':>10} {'RemoteCLIP':>12} {'Delta':>10} {'Rel %':>8}")
print('-' * 54)

for i, name in enumerate(CLASS_NAMES):
    c = clip_ious_on_selected[i]
    r = rc_ious_on_selected[i]
    if c is None or (isinstance(r, float) and np.isnan(r)):
        print(f'{name:<12} {"N/A":>10} {"N/A":>12} {"":>10} {"":>8}')
        continue
    delta = r - c
    rel = (delta / c) * 100 if c > 0 else float('nan')
    print(f'{name:<12} {c:>10.4f} {r:>12.4f} {delta:>+10.4f} {rel:>+7.1f}%')

print('-' * 54)
print(f'{"mIoU":<12} {clip_miou_on_selected:>10.4f} {best_rc_test_miou:>12.4f} '
      f'{best_rc_test_miou - clip_miou_on_selected:>+10.4f} '
      f'{(best_rc_test_miou - clip_miou_on_selected) / clip_miou_on_selected * 100:>+7.1f}%')

Per-class comparison on text_qwen3-4b (CLIP vs RemoteCLIP):

Class              CLIP   RemoteCLIP      Delta    Rel %
------------------------------------------------------
Tree             0.8758       0.8795    +0.0037    +0.4%
Shrub            0.3261       0.3293    +0.0032    +1.0%
Grass            0.8057       0.8044    -0.0013    -0.2%
Crop             0.8159       0.8183    +0.0024    +0.3%
Built-up         0.5893       0.6131    +0.0238    +4.0%
Barren           0.5665       0.5642    -0.0023    -0.4%
Water            0.8994       0.8966    -0.0028    -0.3%
------------------------------------------------------
mIoU             0.6970       0.7008    +0.0038    +0.5%


**Observation:** Per class on `text_qwen3-4b`, the encoder swap improves Tree, Shrub, Crop, and most notably Built-up (+0.0238), while Grass, Barren, and Water decrease slightly. All changes other than Built-up are under 0.004 IoU.

In [33]:
# Persist Ablation 1 results, including the validation-based selection.
ablation1_results = {
    'experiment': 'text_encoder_pretraining',
    'selection_metric': 'val_miou',
    'best_caption_by_val': best_rc_caption,
    'best_caption_val_miou': float(best_rc_val_miou),
    'best_caption_test_miou': float(best_rc_test_miou),
    'remoteclip_val_miou': {
        cap: float(remoteclip_runs[cap]['val_miou']) for cap in CAPTION_COLS
    },
    'phase2_clip_test': {
        cap: {
            'miou':       phase2_clip[cap]['test_miou'],
            'oa':         phase2_clip[cap]['test_oa'],
            'class_ious': phase2_clip[cap]['class_ious'],
        }
        for cap in CAPTION_COLS
    },
    'phase3_remoteclip_test': {
        cap: {
            'miou':       res['miou'],
            'oa':         res['oa'],
            'class_ious': [float(x) if not np.isnan(x) else None for x in res['class_ious']],
        }
        for cap, res in remoteclip_test.items()
    },
}

with open(RESULTS_DIR / 'ablation1_text_encoder.json', 'w') as f:
    json.dump(ablation1_results, f, indent=2)
print(f"Saved to {RESULTS_DIR / 'ablation1_text_encoder.json'}")

Saved to /content/drive/MyDrive/DI725_Project/results/ablation1_text_encoder.json


**Ablation 1 summary:** RemoteCLIP gives a small test improvement over CLIP on the selected caption, with the gain concentrated on a rare, under-segmented class rather than spread across all classes. The effect of the domain-adapted encoder is consistent but modest on this dataset.

## 7. Ablation 2: Loss function

This ablation tests whether replacing weighted cross-entropy with a region-based composite loss improves segmentation. Three region losses are combined with weighted cross-entropy:

- **CE+Dice.** Dice loss optimizes region overlap.
- **CE+Lovász.** Lovász-Softmax is a smooth surrogate for the Jaccard (IoU) index, the evaluation metric itself.
- **CE+Tversky.** Tversky loss generalizes Dice with asymmetric penalties on false positives and false negatives, biased toward recall to help rare classes.

Each region loss is combined with weighted cross-entropy rather than used alone, because region losses can give weak or unstable gradients early in training, while cross-entropy provides a stable per-pixel signal. The composite keeps the training stability of cross-entropy while adding the region term's direct optimization of overlap. Each composite is `alpha * weighted_CE + (1 - alpha) * region_loss` with alpha 0.5.

All three are trained on the fixed caption `text_qwen3-4b` so the comparison isolates the loss. Each variant is checkpointed on validation mIoU, the best loss is selected by validation mIoU, and per-loss test mIoU is reported afterwards as analysis. The weighted cross-entropy result is the Phase 2 multimodal baseline. This ablation varies the choice of region loss and the use of a region term against weighted cross-entropy. It does not separately tune alpha or evaluate the region losses without cross-entropy.

In [34]:
class DiceLoss(nn.Module):
    """Multi-class Dice loss.

    Computes 1 minus the mean per-class Dice coefficient over the classes present
    in the batch. Classes absent from the ground truth are excluded so they do not
    dilute the loss.
    """

    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, target):
        """logits: (B, C, H, W), target: (B, H, W) with class indices."""
        probs = F.softmax(logits, dim=1)
        target_1h = F.one_hot(target, num_classes=probs.size(1)).permute(0, 3, 1, 2).float()

        # Per-class intersection and cardinality, summed over batch and spatial dims.
        dims = (0, 2, 3)
        intersection = (probs * target_1h).sum(dim=dims)
        cardinality = probs.sum(dim=dims) + target_1h.sum(dim=dims)

        # Average only over classes present in the ground truth.
        present = target_1h.sum(dim=dims) > 0
        dice = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        return 1.0 - dice[present].mean()


class TverskyLoss(nn.Module):
    """Multi-class Tversky loss.

    Generalizes Dice with asymmetric weighting of false positives and false
    negatives: Tversky = TP / (TP + alpha * FP + beta * FN). Setting
    alpha = beta = 0.5 recovers Dice. Setting beta greater than alpha biases the
    loss toward recall, which helps rare-class detection.
    """

    def __init__(self, alpha=0.3, beta=0.7, smooth=1.0):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.smooth = smooth

    def forward(self, logits, target):
        """logits: (B, C, H, W), target: (B, H, W) with class indices."""
        probs = F.softmax(logits, dim=1)
        target_1h = F.one_hot(target, num_classes=probs.size(1)).permute(0, 3, 1, 2).float()

        dims = (0, 2, 3)
        tp = (probs * target_1h).sum(dim=dims)
        fp = (probs * (1 - target_1h)).sum(dim=dims)
        fn = ((1 - probs) * target_1h).sum(dim=dims)

        present = target_1h.sum(dim=dims) > 0
        tversky = (tp + self.smooth) / (tp + self.alpha * fp + self.beta * fn + self.smooth)
        return 1.0 - tversky[present].mean()

In [35]:
# Lovász-Softmax, reference implementation adapted to
# the multi-class case. It computes a smooth surrogate for the Jaccard (IoU) loss
# by sorting per-class errors and applying the Lovász extension.

def _lovasz_grad(gt_sorted):
    """Gradient of the Lovász extension of the Jaccard loss for sorted ground truth."""
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jaccard = 1.0 - intersection / union
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard


def _lovasz_softmax_flat(probs, labels, classes='present'):
    """Multi-class Lovász-Softmax on flattened predictions.

    probs:   (P, C) class probabilities.
    labels:  (P,) ground-truth labels.
    classes: 'present' uses only classes present in labels, 'all' uses every class.
    """
    if probs.numel() == 0:
        return probs * 0.0
    C = probs.size(1)
    losses = []
    class_to_sum = list(range(C)) if classes == 'all' else torch.unique(labels).tolist()
    for c in class_to_sum:
        fg = (labels == c).float()
        if fg.sum() == 0:
            continue
        class_pred = probs[:, c]
        errors = (fg - class_pred).abs()
        errors_sorted, perm = torch.sort(errors, 0, descending=True)
        fg_sorted = fg[perm]
        losses.append(torch.dot(errors_sorted, _lovasz_grad(fg_sorted)))
    if len(losses) == 0:
        return probs.sum() * 0.0
    return torch.stack(losses).mean()


class LovaszSoftmaxLoss(nn.Module):
    """Multi-class Lovász-Softmax loss for segmentation logits."""

    def __init__(self, classes='present'):
        super().__init__()
        self.classes = classes

    def forward(self, logits, target):
        """logits: (B, C, H, W), target: (B, H, W)."""
        probs = F.softmax(logits, dim=1)
        B, C, H, W = probs.shape
        probs_flat = probs.permute(0, 2, 3, 1).reshape(-1, C)
        target_flat = target.reshape(-1)
        return _lovasz_softmax_flat(probs_flat, target_flat, classes=self.classes)


class CompositeLoss(nn.Module):
    """Weighted sum of weighted cross-entropy and a region-based loss.

    total = alpha * weighted_CE + (1 - alpha) * region_loss

    Used for CE+Dice, CE+Lovász, and CE+Tversky.
    """

    def __init__(self, ce_weight, region_loss, alpha=0.5):
        super().__init__()
        self.alpha = alpha
        self.ce = nn.CrossEntropyLoss(weight=ce_weight)
        self.region_loss = region_loss

    def forward(self, logits, target):
        return self.alpha * self.ce(logits, target) + (1.0 - self.alpha) * self.region_loss(logits, target)

In [36]:
# Loss hyperparameters
COMPOSITE_ALPHA = 0.5          # weight on weighted CE in each composite loss
TVERSKY_ALPHA   = 0.3          # false-positive weight in Tversky
TVERSKY_BETA    = 0.7          # false-negative weight in Tversky (recall bias)


# Sanity check on a random batch: each loss is finite and produces valid gradients.
def _sanity_loss(name, loss_fn):
    sl = torch.randn(2, NUM_CLASSES, 64, 64, device=device, requires_grad=True)
    st = torch.randint(0, NUM_CLASSES, (2, 64, 64), device=device)
    val = loss_fn(sl, st)
    val.backward()
    ok = sl.grad is not None and torch.isfinite(sl.grad).all().item()
    print(f'{name:<12} value: {val.item():.4f}, gradient OK: {ok}')


_sanity_loss('CE+Dice', CompositeLoss(class_weights_tensor, DiceLoss(), alpha=COMPOSITE_ALPHA).to(device))
_sanity_loss('CE+Lovász', CompositeLoss(class_weights_tensor, LovaszSoftmaxLoss(), alpha=COMPOSITE_ALPHA).to(device))
_sanity_loss('CE+Tversky', CompositeLoss(class_weights_tensor,
                                         TverskyLoss(alpha=TVERSKY_ALPHA, beta=TVERSKY_BETA),
                                         alpha=COMPOSITE_ALPHA).to(device))

CE+Dice      value: 1.6041, gradient OK: True
CE+Lovász    value: 1.5970, gradient OK: True
CE+Tversky   value: 1.5976, gradient OK: True


In [ ]:
# Build the three composite-loss criteria.
dice_criterion = CompositeLoss(class_weights_tensor, DiceLoss(), alpha=COMPOSITE_ALPHA)
lovasz_criterion = CompositeLoss(class_weights_tensor, LovaszSoftmaxLoss(), alpha=COMPOSITE_ALPHA)
tversky_criterion = CompositeLoss(class_weights_tensor,
                                  TverskyLoss(alpha=TVERSKY_ALPHA, beta=TVERSKY_BETA),
                                  alpha=COMPOSITE_ALPHA)

# Train all three composite losses on the fixed caption text_qwen3-4b.
# Each run is checkpointed on best validation mIoU and returns its validation history.
loss_runs = {}

loss_configs = [
    ('ce_dice', dice_criterion,
     {'loss_type': 'ce_dice', 'composite_alpha': COMPOSITE_ALPHA}),
    ('ce_lovasz', lovasz_criterion,
     {'loss_type': 'ce_lovasz', 'composite_alpha': COMPOSITE_ALPHA}),
    ('ce_tversky', tversky_criterion,
     {'loss_type': 'ce_tversky', 'composite_alpha': COMPOSITE_ALPHA,
      'tversky_alpha': TVERSKY_ALPHA, 'tversky_beta': TVERSKY_BETA}),
]

for label, criterion, wandb_extra in loss_configs:
    run_name = f'loss_{label}_text_qwen3_4b'
    print(f'\n{"=" * 60}')
    print(f'Loss: {label}')
    print(f'{"=" * 60}\n')

    hist, val_miou, ckpt = train_multimodal(
        caption_col=BEST_TEXT_CAPTION,
        criterion=criterion,
        run_name=run_name,
        wandb_config={'experiment': 'ablation_2_loss', **wandb_extra},
    )
    save_history(hist, run_name)
    loss_runs[label] = {'history': hist, 'val_miou': val_miou, 'ckpt': ckpt,
                        'criterion': criterion}


Loss: ce_dice



Training loss_ce_dice_text_qwen3_4b for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   0.9627   0.4960   0.5121   0.7752   60.8s   BEST
    2   0.7170   0.4525   0.5638   0.7896   60.1s   BEST
    3   0.6503   0.4123   0.5787   0.7922   61.1s   BEST
    4   0.6204   0.4094   0.5734   0.8162   59.3s       
    5   0.5861   0.4017   0.5872   0.8239   59.9s   BEST
    6   0.5608   0.3650   0.6169   0.8275   60.1s   BEST
    7   0.5566   0.3580   0.6123   0.8203   59.6s       
    8   0.5237   0.3451   0.6212   0.8444   60.5s   BEST
    9   0.5154   0.3594   0.6268   0.8397   59.7s   BEST
   10   0.4998   0.3995   0.6103   0.8160   60.5s       
   11   0.4841   0.3516   0.6865   0.8787   60.3s   BEST
   12   0.4762   0.3969   0.6604   0.8396   59.6s       
   13   0.4590   0.3365   0.6401   0.8546   58.5s       
   14   0.4520   0.3123   0.6621   0.8568   59.8s       
   15   0.4332   0.3276   0.6868   0

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▁▂▂▄▄▄▄▆▅▆▇▇▆▅▇▆█▇▆▇▆▇▇▇██████
val_iou/Built-up,▁▄▄▅▅▆▄▅▆▆▅▇▄▆▇▇▇▇▅▇▇█▇███████
val_iou/Crop,▁▃▅▃▃▅▄▅▅▄▆▂▆▇▆▇▇█▇▇█▇████████
val_iou/Grass,▂▂▁▄▄▅▄▆▅▄▇▅▆▆▇▆▇▇▇▇▇▇█▇██████
val_iou/Shrub,▂▂▂▂▄▂▃▂▃▁▇▅▃▄▅▃▅▅▇▅▆▇▇▆▇██▇██
val_iou/Tree,▁▁▁▂▃▂▂▃▃▂▆▂▄▆▅▆▇▇▆▇▇▇█▇██████
val_iou/Water,▂▄▆▁▁▆▇▄▄▄█▇▇▇█▇██▇███████████
+3,...



Best validation mIoU: 0.7325
Saved history to /content/drive/MyDrive/DI725_Project/results/loss_ce_dice_text_qwen3_4b_history.json

Loss: ce_lovasz



Training loss_ce_lovasz_text_qwen3_4b for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   0.9595   0.5401   0.5521   0.8036   67.1s   BEST
    2   0.7300   0.5128   0.5035   0.7334   66.5s       
    3   0.6872   0.4519   0.5448   0.8007   67.4s       
    4   0.6549   0.4459   0.5920   0.8203   67.1s   BEST
    5   0.6230   0.4421   0.5976   0.8340   66.9s   BEST
    6   0.5954   0.4152   0.6193   0.8289   66.1s   BEST
    7   0.5750   0.3896   0.6277   0.8414   66.6s   BEST
    8   0.5634   0.3924   0.6537   0.8647   67.2s   BEST
    9   0.5429   0.3662   0.6464   0.8375   66.5s       
   10   0.5382   0.3709   0.6703   0.8695   66.2s   BEST
   11   0.5239   0.3705   0.6587   0.8678   66.4s       
   12   0.5153   0.3478   0.6698   0.8691   66.2s       
   13   0.4919   0.3829   0.6489   0.8392   66.0s       
   14   0.4847   0.3443   0.6604   0.8659   66.0s       
   15   0.4742   0.3417   0.6659  

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▅▅▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▃▁▃▂▄▅▃▆▃▅▆▅▁▅▄▇▇▆▄▆█▆█▇█▇▇▇█▇
val_iou/Built-up,▅▁▂▄▄▅▆▆▇▇▆▆█▆▇▇▆▇▇███████████
val_iou/Crop,▁▂▂▂▂▄▄▅▄▅▆▆▄▆▇▇▇▇▇▇█▇████████
val_iou/Grass,▃▁▃▅▅▄▅▇▅▇▇▆▅▇▆▇▇▇▆▇▇▇████████
val_iou/Shrub,▃▁▃▅▅▃▄▆▃▆▅▆▅▅▄▇▇▆▆▇▆▆█▆█▇██▇█
val_iou/Tree,▄▁▅▄▅▅▆▆▇▇▆▇▇▇▇▇▇▇▇█▇▇████████
val_iou/Water,▁▅▂▆▅▇▆▅█▆▅▇▇▇▇▇▆▇██▇█████████
+3,...



Best validation mIoU: 0.7305
Saved history to /content/drive/MyDrive/DI725_Project/results/loss_ce_lovasz_text_qwen3_4b_history.json

Loss: ce_tversky



Training loss_ce_tversky_text_qwen3_4b for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   0.9533   0.4480   0.5157   0.7755   59.8s   BEST
    2   0.6865   0.5086   0.5279   0.7380   59.6s   BEST
    3   0.6469   0.4168   0.5780   0.8115   59.4s   BEST
    4   0.5942   0.4249   0.5581   0.7885   59.9s       
    5   0.5689   0.3907   0.5618   0.7951   58.5s       
    6   0.5544   0.4139   0.6000   0.8085   58.8s   BEST
    7   0.5224   0.3736   0.5987   0.8065   58.9s       
    8   0.5096   0.3576   0.6278   0.8352   59.4s   BEST
    9   0.4930   0.3611   0.6208   0.8346   59.0s       
   10   0.4759   0.3051   0.6432   0.8489   59.6s   BEST
   11   0.4561   0.3677   0.6256   0.8248   58.4s       
   12   0.4469   0.3197   0.6679   0.8564   60.2s   BEST
   13   0.4441   0.3107   0.6733   0.8650   59.9s   BEST
   14   0.4204   0.3042   0.6750   0.8644   59.5s   BEST
   15   0.4099   0.2971   0.6996 

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▅▅▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▂▃▃▃▁▄▂▆▄▅▆▆▇▇▇▆▇▆▇█▇█████████
val_iou/Built-up,▂▃▅▃▁▅▄▆▅▅▅▇▆██▇▇█▇▇██████▇███
val_iou/Crop,▂▂▂▃▄▁▃▄▂▆▅▃▅▆▇▇▇▆▇▇▇▇██▇█████
val_iou/Grass,▁▁▃▂▁▃▃▄▄▅▄▆▆▆▇▇▇▇▇▇▇▇████████
val_iou/Shrub,▃▁▄▃▅▅▄▃▅▄▃▅▆▄▆▇▅▇▆▆▅▆▇▇██████
val_iou/Tree,▃▁▅▃▆▄▅▅▆▇▅▇▇▇▇▇▇▇▇▇▇█████████
val_iou/Water,▁▃▃▄▅▅█▆▆▇▇▇▇▇█████▇██████████
+3,...



Best validation mIoU: 0.7292
Saved history to /content/drive/MyDrive/DI725_Project/results/loss_ce_tversky_text_qwen3_4b_history.json


In [38]:
# Compare all four losses on VALIDATION mIoU, including the weighted-CE baseline
# (Phase 2 model, same caption and recipe), then select the best region loss.
wce_history_path = RESULTS_DIR / f'multimodal_{BEST_TEXT_CAPTION.replace("-", "_")}_history.json'
if wce_history_path.exists():
    with open(wce_history_path) as f:
        wce_val_miou = max(json.load(f)['val_miou'])
else:
    wce_val_miou = None

print('Validation mIoU per loss:')
print(f"{'Loss':<14} {'val mIoU':>10}")
print('-' * 26)
if wce_val_miou is not None:
    print(f'{"weighted_ce":<14} {wce_val_miou:>10.4f}  (Phase 2 baseline)')
for label in loss_runs:
    print(f'{label:<14} {loss_runs[label]["val_miou"]:>10.4f}')

# Select the best REGION loss on validation (the candidates being introduced in
# this ablation). The weighted-CE baseline above is the reference to beat.
best_loss = max(loss_runs, key=lambda k: loss_runs[k]['val_miou'])
best_loss_val_miou = loss_runs[best_loss]['val_miou']

print(f'\nSelected region loss (by validation mIoU): {best_loss}')
print(f'Validation mIoU: {best_loss_val_miou:.4f}')
if wce_val_miou is not None:
    if best_loss_val_miou > wce_val_miou:
        print(f'This exceeds the weighted-CE baseline ({wce_val_miou:.4f}) on validation.')
    else:
        print(f'This does NOT exceed the weighted-CE baseline ({wce_val_miou:.4f}) on validation.')

Validation mIoU per loss:
Loss             val mIoU
--------------------------
weighted_ce        0.6982  (Phase 2 baseline)
ce_dice            0.7325
ce_lovasz          0.7305
ce_tversky         0.7292

Selected region loss (by validation mIoU): ce_dice
Validation mIoU: 0.7325
This exceeds the weighted-CE baseline (0.6982) on validation.


**Observation:** On validation mIoU, all three composite losses exceed the weighted cross-entropy baseline. The three are close to each other, with ce_dice highest, followed by ce_lovasz and ce_tversky. ce_dice is selected as the region loss carried forward.

In [39]:
# Test-set evaluation for the three region losses, plus the weighted-CE baseline.
# Selection above used validation mIoU only. This table is analysis.
# Relative improvement is shown against weighted CE, since only the loss differs
# (same caption, same encoder), so the delta isolates the loss effect.
loss_test = {}
print('Evaluating loss variants on the test set...')
for label, run_info in loss_runs.items():
    class_ious, miou, oa, _ = evaluate_checkpoint(
        caption_col=BEST_TEXT_CAPTION,
        ckpt_path=run_info['ckpt'],
        criterion=run_info['criterion'],
    )
    loss_test[label] = {'class_ious': class_ious, 'miou': miou, 'oa': oa}

# Weighted CE is the Phase 2 multimodal baseline (same caption, same recipe).
# Use the Phase 2 test results directly (loaded earlier as MULTIMODAL_*), so the
# weighted-CE baseline is identical to the reference numbers and to every
# downstream table and saved file.
loss_test['weighted_ce'] = {
    'class_ious': MULTIMODAL_IOUS,
    'miou':       MULTIMODAL_MIOU,
    'oa':         MULTIMODAL_OA,
}

# Print the full table, weighted CE first as the baseline.
# vs WCE columns: absolute and relative mIoU change against weighted CE.
wce_miou = loss_test['weighted_ce']['miou']
order = ['weighted_ce'] + list(loss_runs.keys())
print(f"\n{'Loss':<14} {'test mIoU':>11} {'test OA':>10} {'vs WCE':>10} {'Rel %':>9}")
print('-' * 58)
for label in order:
    m = loss_test[label]
    delta = m['miou'] - wce_miou
    rel   = delta / wce_miou * 100
    marker = '  <- selected' if label == best_loss else ''
    print(f"{label:<14} {m['miou']:>11.4f} {m['oa']:>10.4f} "
          f"{delta:>+10.4f} {rel:>+8.2f}%{marker}")

best_loss_test_miou = loss_test[best_loss]['miou']
best_loss_test_oa   = loss_test[best_loss]['oa']
print(f'\nSelected region loss {best_loss}: test mIoU {best_loss_test_miou:.4f}, test OA {best_loss_test_oa:.4f}')

Evaluating loss variants on the test set...

Loss             test mIoU    test OA     vs WCE     Rel %
----------------------------------------------------------
weighted_ce         0.6970     0.8894    +0.0000    +0.00%
ce_dice             0.7281     0.8984    +0.0311    +4.46%  <- selected
ce_lovasz           0.7275     0.9002    +0.0305    +4.38%
ce_tversky          0.7279     0.9006    +0.0310    +4.44%

Selected region loss ce_dice: test mIoU 0.7281, test OA 0.8984


**Observation:** On the test set, all three composite losses improve over the weighted cross-entropy baseline, by +0.0305 to +0.0311 mIoU. The three are close to each other, with ce_dice highest (+0.0311), then ce_tversky and ce_lovasz within 0.0006 of it. ce_dice, selected on validation, reaches test mIoU 0.7281.

In [40]:
# Four-way per-class comparison: weighted CE (Phase 2) vs the three region losses.
# Baseline is the Phase 2 weighted-CE test result (loss_test['weighted_ce']),
# so this table is consistent with the loss summary table.
print(f'Loss function comparison on {BEST_TEXT_CAPTION}:\n')
print(f"{'Class':<10} {'wCE':>8} {'CE+Dice':>9} {'CE+Lov':>9} {'CE+Tver':>9} "
      f"{'dDice':>8} {'dLov':>8} {'dTver':>8}")
print('-' * 78)

ce_ious = loss_test['weighted_ce']['class_ious']
ce_miou = loss_test['weighted_ce']['miou']

for i, name in enumerate(CLASS_NAMES):
    ce = ce_ious[i]
    dc = loss_test['ce_dice']['class_ious'][i]
    lv = loss_test['ce_lovasz']['class_ious'][i]
    tv = loss_test['ce_tversky']['class_ious'][i]
    if any(v is None or (isinstance(v, float) and np.isnan(v)) for v in [ce, dc, lv, tv]):
        print(f'{name:<10} {"N/A":>8}')
        continue
    print(f'{name:<10} {ce:>8.4f} {dc:>9.4f} {lv:>9.4f} {tv:>9.4f} '
          f'{dc-ce:>+8.4f} {lv-ce:>+8.4f} {tv-ce:>+8.4f}')

print('-' * 78)
miou_dc = loss_test['ce_dice']['miou']
miou_lv = loss_test['ce_lovasz']['miou']
miou_tv = loss_test['ce_tversky']['miou']
print(f'{"mIoU":<10} {ce_miou:>8.4f} {miou_dc:>9.4f} {miou_lv:>9.4f} {miou_tv:>9.4f} '
      f'{miou_dc-ce_miou:>+8.4f} {miou_lv-ce_miou:>+8.4f} {miou_tv-ce_miou:>+8.4f}')

Loss function comparison on text_qwen3-4b:

Class           wCE   CE+Dice    CE+Lov   CE+Tver    dDice     dLov    dTver
------------------------------------------------------------------------------
Tree         0.8758    0.8832    0.8760    0.8840  +0.0074  +0.0001  +0.0082
Shrub        0.3261    0.3716    0.3247    0.3682  +0.0455  -0.0014  +0.0421
Grass        0.8057    0.8190    0.8298    0.8237  +0.0133  +0.0242  +0.0180
Crop         0.8159    0.8137    0.8176    0.8192  -0.0023  +0.0017  +0.0032
Built-up     0.5893    0.7116    0.7096    0.6801  +0.1223  +0.1204  +0.0908
Barren       0.5665    0.5768    0.6192    0.6054  +0.0103  +0.0527  +0.0389
Water        0.8994    0.9206    0.9154    0.9148  +0.0212  +0.0160  +0.0154
------------------------------------------------------------------------------
mIoU         0.6970    0.7281    0.7275    0.7279  +0.0311  +0.0305  +0.0310


**Observation:** Per class, the composite losses improve over weighted cross-entropy on most classes, with the largest gain on Built-up under all three (ce_dice +0.1223, ce_lovasz +0.1204, ce_tversky +0.0908). Shrub also improves substantially under ce_dice and ce_tversky. The remaining classes change by smaller amounts, mostly positive, with only Crop slightly down under ce_dice and Shrub slightly down under ce_lovasz.

In [41]:
# Persist Ablation 2 results, including the validation-based selection.
# The weighted-CE baseline is the Phase 2 multimodal test result, so every
# weighted-CE number in this file is from one consistent source and matches
# the comparison tables above.
ablation2_results = {
    'experiment':       'loss_function',
    'caption_col':      BEST_TEXT_CAPTION,
    'selection_metric': 'val_miou',
    'composite_alpha':  COMPOSITE_ALPHA,
    'tversky_alpha':    TVERSKY_ALPHA,
    'tversky_beta':     TVERSKY_BETA,
    'best_loss_by_val':     best_loss,
    'best_loss_val_miou':   float(best_loss_val_miou),
    'best_loss_test_miou':  float(best_loss_test_miou),
    'weighted_ce_baseline': {
        'test_miou':  loss_test['weighted_ce']['miou'],
        'test_oa':    loss_test['weighted_ce']['oa'],
        'class_ious': [float(x) if x is not None and not np.isnan(x) else None
                       for x in loss_test['weighted_ce']['class_ious']],
    },
    'loss_val_miou': {
        label: float(loss_runs[label]['val_miou']) for label in loss_runs
    },
    'loss_test': {
        label: {
            'miou':       res['miou'],
            'oa':         res['oa'],
            'class_ious': [float(x) if x is not None and not np.isnan(x) else None
                           for x in res['class_ious']],
        }
        for label, res in loss_test.items()
    },
}

with open(RESULTS_DIR / 'ablation2_loss.json', 'w') as f:
    json.dump(ablation2_results, f, indent=2)
print(f"Saved to {RESULTS_DIR / 'ablation2_loss.json'}")

Saved to /content/drive/MyDrive/DI725_Project/results/ablation2_loss.json


**Ablation 2 summary:** All three composite losses improve over weighted cross-entropy on the test set, and they perform very similarly to each other. The selected loss, ce_dice, gives the larger of the two ablation gains, with the improvement concentrated on a rare, under-segmented class rather than spread across all classes.

## 8. Additivity check

Ablation 1 and Ablation 2 each improve a different part of the model: the text encoder and the loss function. This section tests whether the two combine additively. A single model is trained on the anchor caption `text_qwen3-4b` with both changes at once: the RemoteCLIP text encoder and the loss selected on validation in Ablation 2.

The interaction term is the combined gain minus the sum of the two individual gains: near zero means additive, positive means super-additive, and negative means sub-additive. The best Phase 3 configuration is identified at the end by comparing the single-component models and the combined model.

In [42]:
# The combined model applies both ablation winners on the anchor caption:
# RemoteCLIP text encoder (Ablation 1) and the validation-selected best loss
# (Ablation 2). The best loss is read from Ablation 2's selection, not hardcoded.
combined_model_factory = lambda: UNetFormerRemoteCLIP(num_classes=NUM_CLASSES)
combined_criterion = loss_runs[best_loss]['criterion']

print('Combined model configuration:')
print(f'  Text encoder : RemoteCLIP (ViT-B-32)')
print(f'  Loss         : {best_loss} (validation-selected in Ablation 2)')
print(f'  Caption      : {BEST_TEXT_CAPTION}')
print(f'  Recipe       : {NUM_EPOCHS} epochs, AdamW lr {LR}, weight decay {WEIGHT_DECAY}')

Combined model configuration:
  Text encoder : RemoteCLIP (ViT-B-32)
  Loss         : ce_dice (validation-selected in Ablation 2)
  Caption      : text_qwen3-4b
  Recipe       : 30 epochs, AdamW lr 0.0006, weight decay 0.01


In [ ]:
# Train the combined model.
print(f'{"=" * 60}')
print(f'Training combined model: RemoteCLIP + {best_loss} on {BEST_TEXT_CAPTION}')
print(f'{"=" * 60}\n')

hist_combined, val_miou_combined, ckpt_combined = train_multimodal(
    caption_col=BEST_TEXT_CAPTION,
    model_factory=combined_model_factory,
    criterion=combined_criterion,
    run_name='additivity_remoteclip_bestloss',
    wandb_config={
        'experiment':   'additivity_check',
        'text_encoder': 'remoteclip',
        'loss_type':    best_loss,
    },
)
save_history(hist_combined, 'additivity_remoteclip_bestloss')

# Test-set evaluation of the combined model.
class_ious_combined, miou_combined, oa_combined, _ = evaluate_checkpoint(
    caption_col=BEST_TEXT_CAPTION,
    ckpt_path=ckpt_combined,
    model_factory=combined_model_factory,
    criterion=combined_criterion,
)

print(f'\nCombined model on test set:')
print(f'  validation mIoU = {val_miou_combined:.4f}')
print(f'  test mIoU       = {miou_combined:.4f}')
print(f'  test OA         = {oa_combined:.4f}')

Training combined model: RemoteCLIP + ce_dice on text_qwen3-4b



RemoteCLIP ViT-B-32 loaded: <All keys matched successfully>


Training additivity_remoteclip_bestloss for 30 epochs...
Epoch   T_Loss   V_Loss   V_mIoU     V_OA    Time       
-------------------------------------------------------
    1   0.9581   0.4843   0.5380   0.7877   58.5s   BEST
    2   0.7135   0.4524   0.5567   0.7880   60.1s   BEST
    3   0.6520   0.4014   0.6111   0.8244   60.9s   BEST
    4   0.6058   0.4240   0.5813   0.7866   59.3s       
    5   0.5794   0.4048   0.5882   0.8097   59.2s       
    6   0.5627   0.3757   0.5882   0.8315   59.7s       
    7   0.5631   0.4050   0.5801   0.7983   59.0s       
    8   0.5164   0.3896   0.5732   0.8034   59.0s       
    9   0.5131   0.3425   0.6232   0.8392   59.6s   BEST
   10   0.4980   0.3891   0.6048   0.7986   58.5s       
   11   0.4801   0.3633   0.6297   0.8097   59.7s   BEST
   12   0.4649   0.3228   0.6487   0.8490   59.6s   BEST
   13   0.4562   0.3156   0.6635   0.8606   60.2s   BEST
   14   0.4380   0.3032   0.6861   0.8739   59.7s   BEST
   15   0.4272   0.3203   0.6758

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train_loss,█▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_iou/Barren,▃▂▅▃▂▄▂▁▄▁▃▆▅▇▆█▇▇██▆█▅▇▇███▇▇
val_iou/Built-up,▁▄▆▅▅▄▃▆▄▆▇▅▆▇▆▇██▇█▇█▇▇██████
val_iou/Crop,▃▃▄▄▃▅▁▅▅▅▆▇▆▇▆▇▇▇▇▇█▇▇███████
val_iou/Grass,▁▁▄▁▄▄▃▃▆▂▄▅▆▇▆▆▆▆▇█▇█▇▇██████
val_iou/Shrub,▂▂▃▁▂▄▄▂▃▂▁▃▅▅▆▄▅▃▆▆▆▆▇▇▆█▇███
val_iou/Tree,▁▁▄▂▃▃▂▃▃▃▂▄▆▆▆▅▇▅▇▇▇▇▇███████
val_iou/Water,▃▂▅▆▅▁▇▁▇▇██▇▇▇███▇▇██▇███████
+3,...



Best validation mIoU: 0.7308
Saved history to /content/drive/MyDrive/DI725_Project/results/additivity_remoteclip_bestloss_history.json


RemoteCLIP ViT-B-32 loaded: <All keys matched successfully>

Combined model on test set:
  validation mIoU = 0.7308
  test mIoU       = 0.7347
  test OA         = 0.9024


In [45]:
# Additivity analysis. Single-component test mIoU values come from the
# validation-selected configurations in Ablations 1 and 2. The baseline is the
# Phase 2 weighted-CE test result, so every number here reconciles with the
# loss comparison table.
baseline_miou         = loss_test['weighted_ce']['miou']            # CLIP, weighted CE
remoteclip_alone_miou = remoteclip_test[BEST_TEXT_CAPTION]['miou']  # RemoteCLIP, wCE, anchor caption
bestloss_alone_miou   = loss_test[best_loss]['miou']               # CLIP, best loss

# Single-component deltas versus the multimodal baseline.
delta_remoteclip = remoteclip_alone_miou - baseline_miou
delta_bestloss   = bestloss_alone_miou   - baseline_miou
delta_combined   = miou_combined         - baseline_miou

# Interaction: combined gain minus the sum of the two single-component gains.
# Negative means sub-additive, positive means super-additive.
interaction_term = delta_combined - (delta_remoteclip + delta_bestloss)

print('Additivity analysis:\n')
print(f"{'Configuration':<34} {'test mIoU':>11} {'d vs baseline':>15}")
print('-' * 62)
print(f"{'Multimodal baseline (CLIP, wCE)':<34} {baseline_miou:>11.4f} {'--':>15}")
print(f"{'RemoteCLIP alone':<34} {remoteclip_alone_miou:>11.4f} {delta_remoteclip:>+15.4f}")
print(f"{best_loss + ' alone':<34} {bestloss_alone_miou:>11.4f} {delta_bestloss:>+15.4f}")
print(f"{'Combined (RemoteCLIP + ' + best_loss + ')':<34} {miou_combined:>11.4f} {delta_combined:>+15.4f}")
print('-' * 62)
print(f"{'Interaction term':<34} {'':>11} {interaction_term:>+15.4f}")
print()
print(f'Relative improvement vs baseline (mIoU {baseline_miou:.4f}):')
print(f'  {"RemoteCLIP alone":<16}: {delta_remoteclip / baseline_miou * 100:+.2f}%')
print(f'  {best_loss + " alone":<16}: {delta_bestloss / baseline_miou * 100:+.2f}%')
print(f'  {"Combined":<16}: {delta_combined / baseline_miou * 100:+.2f}%')

Additivity analysis:

Configuration                        test mIoU   d vs baseline
--------------------------------------------------------------
Multimodal baseline (CLIP, wCE)         0.6970              --
RemoteCLIP alone                        0.7008         +0.0038
ce_dice alone                           0.7281         +0.0311
Combined (RemoteCLIP + ce_dice)         0.7347         +0.0378
--------------------------------------------------------------
Interaction term                                       +0.0028

Relative improvement vs baseline (mIoU 0.6970):
  RemoteCLIP alone: +0.55%
  ce_dice alone   : +4.46%
  Combined        : +5.42%


**Observation:** Combining the RemoteCLIP encoder with the ce_dice loss reaches test mIoU 0.7347, above either single component alone. The interaction term is +0.0028, slightly positive, so the combined gain is marginally larger than the sum of the two individual gains. The two improvements stack approximately additively.

In [46]:
# Per-class IoU across all configurations: vision-only baseline, multimodal
# baseline (weighted CE), RemoteCLIP alone, best-loss alone, and the combined model.
# RemoteCLIP and multimodal columns use the same sources as the additivity table,
# so every number reconciles across the section.
rc_ious_for_table       = remoteclip_test[BEST_TEXT_CAPTION]['class_ious']
bestloss_ious_for_table = loss_test[best_loss]['class_ious']
mm_ious_for_table       = loss_test['weighted_ce']['class_ious']
mm_miou                 = loss_test['weighted_ce']['miou']

print(f'Per-class IoU across configurations:\n')
print(f"{'Class':<10} {'VisOnly':>9} {'Multimod':>9} {'RemoteCLIP':>11} "
      f"{best_loss:>12} {'Combined':>10}")
print('-' * 64)
for i, name in enumerate(CLASS_NAMES):
    bl = BASELINE_IOUS[i]
    mm = mm_ious_for_table[i]
    rc = rc_ious_for_table[i]
    lv = bestloss_ious_for_table[i]
    cm = class_ious_combined[i]
    values = [bl, mm, rc, lv, cm]
    if any(v is None or (isinstance(v, float) and np.isnan(v)) for v in values):
        print(f'{name:<10} {"N/A":>9}')
        continue
    print(f'{name:<10} {bl:>9.4f} {mm:>9.4f} {rc:>11.4f} {lv:>12.4f} {cm:>10.4f}')
print('-' * 64)
print(f'{"mIoU":<10} {BASELINE_MIOU:>9.4f} {mm_miou:>9.4f} '
      f'{remoteclip_alone_miou:>11.4f} {bestloss_alone_miou:>12.4f} {miou_combined:>10.4f}')

Per-class IoU across configurations:

Class        VisOnly  Multimod  RemoteCLIP      ce_dice   Combined
----------------------------------------------------------------
Tree          0.8608    0.8758      0.8795       0.8832     0.8830
Shrub         0.2476    0.3261      0.3293       0.3716     0.3708
Grass         0.7631    0.8057      0.8044       0.8190     0.8279
Crop          0.7416    0.8159      0.8183       0.8137     0.8228
Built-up      0.5631    0.5893      0.6131       0.7116     0.7137
Barren        0.4869    0.5665      0.5642       0.5768     0.6068
Water         0.8784    0.8994      0.8966       0.9206     0.9182
----------------------------------------------------------------
mIoU          0.6488    0.6970      0.7008       0.7281     0.7347


**Observation:** Across the five configurations, mIoU rises steadily from vision-only (0.6488) to the combined model (0.7347). The largest cumulative gain is on Built-up, which rises from 0.5631 to 0.7137. Barren and Shrub also improve substantially across the progression, while the already-strong classes (Tree, Grass, Crop, Water) improve by smaller amounts. The combined model is the best or near-best configuration for most classes.

In [ ]:
# Save best Phase 3 test predictions for 06_qualitative_analysis.ipynb.
# Best Phase 3 model is the combined config: RemoteCLIP encoder + ce_dice loss.
# Same unshuffled test_df order as nb03/nb04, so all .npy files are index-aligned.
PREDICTIONS_DIR = PROJECT_DIR / 'predictions'
PREDICTIONS_DIR.mkdir(exist_ok=True)

_, _, pred_test_loader = make_loaders(BEST_TEXT_CAPTION)

model_p3 = combined_model_factory().to(device)
model_p3.load_state_dict(torch.load(ckpt_combined))
model_p3.eval()

all_preds = []
with torch.no_grad():
    for imgs, masks, captions in pred_test_loader:
        imgs = imgs.to(device)
        preds = model_p3(imgs, list(captions)).argmax(dim=1)
        all_preds.append(preds.cpu().numpy().astype(np.uint8))

preds_array = np.concatenate(all_preds, axis=0)
np.save(PREDICTIONS_DIR / 'phase3_best_masks.npy', preds_array)
print(f'Saved phase3_best_masks.npy: {preds_array.shape}')
print('count matches test split:', len(preds_array) == len(test_df))
print('class ids:', np.unique(preds_array))

del model_p3
torch.cuda.empty_cache()

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name swsl_resnet18 to current resnet18.fb_swsl_ig1b_ft_in1k.
  model = create_fn(


RemoteCLIP ViT-B-32 loaded: <All keys matched successfully>
Saved phase3_best_masks.npy: (1500, 256, 256)
count matches test split: True
class ids: [0 1 2 3 4 5 6]


In [ ]:
# Persist the additivity-check results.
# baseline_miou uses loss_test['weighted_ce'] (the Phase 2 weighted-CE test
# result) so every saved file reports the same baseline (matches
# ablation2_loss.json and the tables above).
additivity_results = {
    'experiment':  'additivity_check',
    'caption_col': BEST_TEXT_CAPTION,
    'encoder':     'remoteclip',
    'loss':        best_loss,
    'baseline_miou':          loss_test['weighted_ce']['miou'],
    'remoteclip_alone_miou':  remoteclip_alone_miou,
    'bestloss_alone_miou':    bestloss_alone_miou,
    'combined': {
        'val_miou':   float(val_miou_combined),
        'test_miou':  float(miou_combined),
        'test_oa':    float(oa_combined),
        'class_ious': [float(x) if x is not None and not np.isnan(x) else None
                       for x in class_ious_combined],
    },
    'additivity': {
        'delta_remoteclip': float(delta_remoteclip),
        'delta_bestloss':   float(delta_bestloss),
        'delta_combined':   float(delta_combined),
        'interaction_term': float(interaction_term),
    },
}

with open(RESULTS_DIR / 'additivity_check.json', 'w') as f:
    json.dump(additivity_results, f, indent=2)
print(f"Saved to {RESULTS_DIR / 'additivity_check.json'}")

Saved to /content/drive/MyDrive/DI725_Project/results/additivity_check.json


## 9. Phase 3 Summary


In [47]:
# Identify the best Phase 3 configuration among the validation-selected
# single-component models and the combined model, by test mIoU.
phase3_candidates = {
    'RemoteCLIP alone':   remoteclip_alone_miou,
    f'{best_loss} alone': bestloss_alone_miou,
    'Combined (both)':    miou_combined,
}
best_phase3_name = max(phase3_candidates, key=phase3_candidates.get)
best_phase3_miou = phase3_candidates[best_phase3_name]

# Baseline from the same source as every other table (Phase 2 weighted-CE test result).
baseline_miou = loss_test['weighted_ce']['miou']
baseline_oa   = loss_test['weighted_ce']['oa']

# Unified results table.
unified_rows = [
    ('Multimodal baseline',      baseline_miou,          baseline_oa),
    ('Ablation 1: RemoteCLIP',   remoteclip_alone_miou,  best_rc_test_oa),
    (f'Ablation 2: {best_loss}', bestloss_alone_miou,    best_loss_test_oa),
    ('Combined (both)',          miou_combined,          oa_combined),
]
df = pd.DataFrame(unified_rows, columns=['Configuration', 'test mIoU', 'test OA'])
df['d vs baseline']     = df['test mIoU'] - baseline_miou
df['Rel % vs baseline'] = df['d vs baseline'] / baseline_miou * 100

# The baseline row is the anchor, so its delta columns are blank.
baseline_mask = df['Configuration'] == 'Multimodal baseline'
df.loc[baseline_mask, ['d vs baseline', 'Rel % vs baseline']] = pd.NA

# Highlight the best Phase 3 configuration (rows 1 to 3, excluding the baseline).
best_idx = 1 + df.iloc[1:4]['test mIoU'].values.argmax()

def highlight_best(row):
    if row.name == best_idx:
        return ['font-weight: bold'] * len(row)
    return [''] * len(row)

styled = (
    df.style
      .format({'test mIoU':         '{:.4f}',
               'test OA':           '{:.4f}',
               'd vs baseline':     '{:+.4f}',
               'Rel % vs baseline': '{:+.2f}%'},
              na_rep='--')
      .apply(highlight_best, axis=1)
      .set_caption(f'Phase 3 unified results. Best Phase 3 model: {df.iloc[best_idx]["Configuration"]}')
      .hide(axis='index')
)
styled

Configuration,test mIoU,test OA,d vs baseline,Rel % vs baseline
Multimodal baseline,0.6970,0.8894,--,--
Ablation 1: RemoteCLIP,0.7008,0.8904,+0.0038,+0.55%
Ablation 2: ce_dice,0.7281,0.8984,+0.0311,+4.46%
Combined (both),0.7347,0.9024,+0.0378,+5.42%


**Observation:** Phase 3 evaluated two independent improvements to the multimodal model and their combination, all on the anchor caption `text_qwen3-4b`, selected on validation and reported on test.

- **Encoder (Ablation 1):** Replacing CLIP with the domain-adapted RemoteCLIP text encoder gives a small test gain (mIoU 0.6970 to 0.7008, +0.0038), concentrated almost entirely on the Built-up class.
- **Loss (Ablation 2):** Replacing weighted cross-entropy with a composite loss gives a larger gain. All three composites improve over the baseline and perform similarly. ce_dice is selected on validation and reaches test mIoU 0.7281 (+0.0311), again with the largest per-class improvement on Built-up.
- **Combination (Additivity):** Applying both at once reaches test mIoU 0.7347 (+0.0378 over the multimodal baseline). The interaction term is +0.0028, so the two gains combine approximately additively.

The best Phase 3 configuration is the combined model (RemoteCLIP encoder with the ce_dice composite loss), at test mIoU 0.7347, +5.42% over the multimodal baseline. The gains across both ablations concentrate on rare, under-segmented classes rather than the already-strong majority classes.

## 10. Overall project summary

Across the full project, segmentation improved from the vision-only baseline through the addition of text guidance and the Phase 3 refinements. The table below shows the cumulative improvement at each configuration, anchored to the vision-only model.

In [49]:
# Project-arc summary: cumulative improvement from the vision-only baseline.
# This view is anchored to the vision-only model to show the full-project gain,
# complementing the multimodal-anchored ablation table above.
arc_rows = [
    ('Vision-only (UNetFormer)',                  BASELINE_MIOU,          BASELINE_OA),
    ('+ CLIP text, weighted CE',                  baseline_miou,          baseline_oa),
    ('+ RemoteCLIP text, weighted CE',            remoteclip_alone_miou,  best_rc_test_oa),
    ('+ CLIP text, CE+Dice loss',                 bestloss_alone_miou,    best_loss_test_oa),
    ('+ RemoteCLIP text, CE+Dice loss',           miou_combined,          oa_combined),
]
arc = pd.DataFrame(arc_rows, columns=['Configuration', 'test mIoU', 'test OA'])
arc['d vs vision-only']     = arc['test mIoU'] - BASELINE_MIOU
arc['Rel % vs vision-only'] = arc['d vs vision-only'] / BASELINE_MIOU * 100

# Vision-only is the anchor, so its delta columns are blank.
vis_mask = arc['Configuration'] == 'Vision-only (UNetFormer)'
arc.loc[vis_mask, ['d vs vision-only', 'Rel % vs vision-only']] = pd.NA

arc_styled = (
    arc.style
       .format({'test mIoU':            '{:.4f}',
                'test OA':              '{:.4f}',
                'd vs vision-only':     '{:+.4f}',
                'Rel % vs vision-only': '{:+.2f}%'},
               na_rep='--')
       .set_caption('Project-arc summary: cumulative improvement from the vision-only baseline.')
       .hide(axis='index')
)
arc_styled

Configuration,test mIoU,test OA,d vs vision-only,Rel % vs vision-only
Vision-only (UNetFormer),0.6488,0.8621,--,--
"+ CLIP text, weighted CE",0.6970,0.8894,+0.0482,+7.42%
"+ RemoteCLIP text, weighted CE",0.7008,0.8904,+0.0520,+8.01%
"+ CLIP text, CE+Dice loss",0.7281,0.8984,+0.0793,+12.22%
"+ RemoteCLIP text, CE+Dice loss",0.7347,0.9024,+0.0859,+13.25%


**Observations:** The combined Phase 3 model reaches test mIoU 0.7347, a +13.24% relative improvement over the vision-only baseline. The improvement comes in two stages: adding a frozen text encoder with gated cross-attention to the vision-only model accounts for the larger share of the gain, and the Phase 3 refinements (the domain-adapted RemoteCLIP encoder and the CE+Dice composite loss) add a further improvement on top.

A consistent pattern holds across the whole project: the gains concentrate on the rare, under-segmented classes, most clearly Built-up, while the high-coverage classes were already well-segmented by the vision-only model and change little. This is the expected behavior when the additions target class imbalance, through either richer text guidance for rare semantic categories or a region-based loss that weights small classes more directly than per-pixel cross-entropy. Overall accuracy moves much less than mIoU across the configurations, consistent with the gains being driven by minority classes rather than the dominant ones.